# 04_damage_robust_quant

Structured notebook generated from the supplied script.

## Header / Overview

In [ ]:
"""
04_damage_robust_quant.py
─────────────────────────
Sections 3 and 4 of the Mechanistic Interpretability assignment.

Section 3: Representation Damage and Causal Importance
  - Neuron ranking (ℓ2 difference, KL divergence)
  - Spectral analysis (singular values, principal angles, SDS)
  - Ablation study (top-5 damaged vs random-5)
  - Alignment: quantisation damage vs perplexity impact

Section 4: Mechanistic Explanation and Robust Quantisation
  - Jacobian norms and Fisher-style importance
  - Failure mode tests (low-variance collapse, sparsity fragility, subspace rotation)
  - Subspace-preserving quantisation and final comparison

Paste class/function definitions from notebooks 01–03 above this file:
  ActivationNormalizer, SparseAutoencoder, load_frozen_model, load_sae,
  collect_held_out, CaptureHook, PatchingHook, make_quantise_fn,
  _uniform_quantise, _calibrate, _perplexity_clean, _perplexity_patched,
  _collect_bottleneck_pairs, _compute_subspace, _sds, _cka, _mse
"""

## 1. Config

In [ ]:
# ── 1. Config ─────────────────────────────────────────────────────────────────
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr
from typing import Optional, Callable
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
LAYER_IDX  = 3
D_MODEL    = 768
M          = 512
K          = M // 10
EVAL_BATCH = 32
SEQ_LEN    = 128

NOTEBOOK01_DIR = Path("/kaggle/input/notebooks/codemtc/pipeline-setup/activations")
NOTEBOOK02_DIR = Path("/kaggle/input/notebooks/codemtc/sae-training-m512")
OUT_DIR        = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

HELD_OUT_SEQS     = 2_000
METRIC_SEQS       = 400      # sequences for metric computation (~50k tokens)
JACOBIAN_BATCHES  = 10       # forward-backward passes for Jacobian estimation
TOP_N_FEATURES    = 5        # features to ablate individually
RANDOM_N_FEATURES = 5
SDS_K             = 64       # primary k for spectral analysis

### Classes from old notebooks

In [ ]:
class ActivationNormalizer:
    """
    Per-dimension (channel-wise) z-score normalizer.

    Fit on the debug pass (~200k tokens); save and reuse for the full extraction
    and at eval time. Consistent normalization is critical: the SAE learns a
    dictionary in the normalized space, so test-time patching must use the same
    mean/std.

    Why per-dimension?  distilgpt2 layer-3 output dimensions have very different
    scales — some are near-zero throughout the dataset, others span ±10+.
    Global (scalar) normalization leaves that structure intact, making the SAE
    learn an uneven dictionary. Per-dimension normalization is what Anthropic and
    Nanda's public SAE implementations both use.
    """

    def __init__(self):
        self.mean: Optional[torch.Tensor] = None
        self.std:  Optional[torch.Tensor] = None

    def fit(self, acts: torch.Tensor, eps: float = 1e-6):
        """acts: (N, D_MODEL), any dtype."""
        a = acts.float()
        self.mean = a.mean(dim=0)                    # (D_MODEL,)
        self.std  = a.std(dim=0).clamp(min=eps)      # (D_MODEL,)
        print(
            f"  normalizer: mean.norm={self.mean.norm():.3f}  "
            f"std.mean={self.std.mean():.4f}  "
            f"std.min={self.std.min():.6f}"
        )

    def __call__(self, acts: torch.Tensor) -> torch.Tensor:
        """Return normalized activations in the same dtype as input."""
        dtype = acts.dtype
        a = acts.float()
        out = (a - self.mean.to(a.device)) / self.std.to(a.device)
        return out.to(dtype)

    def inverse(self, acts: torch.Tensor) -> torch.Tensor:
        dtype = acts.dtype
        a = acts.float()
        out = a * self.std.to(a.device) + self.mean.to(a.device)
        return out.to(dtype)

    def save(self, path: Path):
        torch.save({"mean": self.mean, "std": self.std}, path)
        print(f"  normalizer saved → {path}")

    @classmethod
    def load(cls, path: Path) -> "ActivationNormalizer":
        n = cls()
        ckpt = torch.load(path, map_location="cpu")
        n.mean, n.std = ckpt["mean"], ckpt["std"]
        return n

In [ ]:
class SparseAutoencoder(nn.Module):
    """
    Top-k Sparse Autoencoder.

    Architecture (tied-bias formulation, standard in mech interp):
        z    = topk( ReLU( W_enc @ (x - b_dec) + b_enc ) )
        x̂   = W_dec @ z + b_dec
        loss = MSE(x, x̂)

    The pre-encoder bias (b_dec) is subtracted before encoding and added
    back after decoding. This centers the input around the decoder's
    natural origin so the encoder learns directions, not offsets.

    Decoder columns (dictionary atoms) are kept at unit norm after every
    gradient step. Without this constraint, the trivially optimal solution
    is to make decoder atoms very large and encoder weights very small,
    which achieves low MSE without learning anything meaningful.

    Top-k vs L1
    ───────────
    The assignment specifies top-k sparsity rather than the L1 penalty used
    in Anthropic's original paper. Top-k has two advantages here: (1) it
    guarantees exactly K active features per token rather than varying
    sparsity, making L0 a constant diagnostic rather than a tunable one;
    (2) it removes the L1 coefficient as a hyperparameter to tune.
    """

    def __init__(self, d_in: int = D_MODEL, m: int = M, k: int = K):
        super().__init__()
        self.d_in = d_in
        self.m    = m
        self.k    = k

        # Encoder weights and bias
        self.W_enc = nn.Parameter(torch.empty(d_in, m))
        self.b_enc = nn.Parameter(torch.zeros(m))

        # Decoder weights (columns = dictionary atoms) and shared bias
        self.W_dec = nn.Parameter(torch.empty(d_in, m))
        self.b_dec = nn.Parameter(torch.zeros(d_in))

        self._init_weights()

    def _init_weights(self):
        nn.init.kaiming_uniform_(self.W_enc, nonlinearity="relu")
        # Initialize decoder as encoder transpose then normalize.
        # This gives a reasonable starting point where encoder and decoder
        # are approximately inverses of each other.
        with torch.no_grad():
            self.W_dec.data = self.W_enc.data.T.clone()
            self._normalize_decoder()

    @torch.no_grad()
    def _normalize_decoder(self):
        """Normalize each column of W_dec to unit norm in-place."""
        # W_dec: (m, d_in) — normalize along d_in dimension (dim=1)
        norms = self.W_dec.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self.W_dec.data /= norms

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, d_in) → z: (batch, m)
        z has exactly K non-zero entries per row.
        """
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc  # (batch, m)
        acts = F.relu(pre)

        # Hard top-k: scatter the k largest values, zero the rest.
        # sorted=False is faster and order doesn't matter here.
        topk_vals, topk_idx = acts.topk(self.k, dim=-1, sorted=False)
        z = torch.zeros_like(acts)
        z.scatter_(-1, topk_idx, topk_vals)
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """z: (batch, m) → x̂: (batch, d_in)"""
        return z @ self.W_dec + self.b_dec

    def forward(self, x: torch.Tensor):
        z    = self.encode(x)
        x_hat = self.decode(z)
        loss  = F.mse_loss(x_hat, x)
        return loss, z, x_hat

### Functions from old notebooks

In [ ]:
def load_frozen_model(device: str = DEVICE):
    """
    Load distilgpt2 with all parameters frozen and in eval mode.
    Keep weights in fp32 for numerical correctness; activations are cast to
    fp16 in the hook to save RAM during extraction.
    """
    tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
    tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

    model = AutoModelForCausalLM.from_pretrained(
        "distilgpt2", torch_dtype=torch.float32
    ).to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    return model, tokenizer

def load_sae(path: str | Path, device: str = DEVICE) -> SparseAutoencoder:
    """Load a saved SAE checkpoint. Use this at the top of Notebooks 03 & 04."""
    ckpt = torch.load(path, map_location=device)
    cfg  = ckpt["config"]
    sae  = SparseAutoencoder(d_in=cfg["d_in"], m=cfg["m"], k=cfg["k"]).to(device)
    sae.load_state_dict(ckpt["model_state"])
    sae.eval()
    print(f"  loaded SAE m={cfg['m']} k={cfg['k']} from step {ckpt['step']:,}")
    return sae

def _token_windows(tokenizer, max_seqs: int, skip_tokens: int = 12_000_000):
    """
    Yield non-overlapping 128-token windows, skipping the first `skip_tokens`
    tokens so the held-out set does not overlap with the 10M training tokens.
    """
    ds = load_dataset(
        "Skylion007/openwebtext", split="train", streaming=True
    )
    carry   = torch.empty(0, dtype=torch.long)
    skipped = 0
    yielded = 0

    for ex in ds:
        if yielded >= max_seqs:
            break
        ids = tokenizer.encode(ex["text"], add_special_tokens=False,
                               return_tensors="pt").squeeze(0)
        ids = torch.cat([carry, ids, torch.tensor([tokenizer.eos_token_id])])

        n = len(ids) // SEQ_LEN
        for i in range(n):
            window = ids[i * SEQ_LEN : (i + 1) * SEQ_LEN]
            if skipped < skip_tokens:
                skipped += SEQ_LEN
                continue
            if yielded >= max_seqs:
                break
            yield window
            yielded += 1
        carry = ids[n * SEQ_LEN :]

@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids

class CaptureHook:
    """
    Captures layer-3 hidden states without modifying them.
    Used to collect full-precision activations for metric computation.
    """
    def __init__(self):
        self.activations: list[torch.Tensor] = []
        self._handle = None

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            lambda m, inp, out: self.activations.append(
                out[0].detach().cpu().to(torch.float16)
            )
        )
        return self

    def pop(self) -> torch.Tensor:
        out = torch.cat(self.activations, dim=0)
        self.activations.clear()
        return out

    def remove(self):
        if self._handle:
            self._handle.remove()

class PatchingHook:
    """
    Replaces layer-3 hidden states with SAE-reconstructed (optionally
    quantised) activations. The patched activations flow through layers 4-5
    and the LM head, so the final perplexity reflects the information loss
    from quantisation.

    Also stores captured bottleneck activations (before and after quantisation)
    for SDS and CKA computation.
    """
    def __init__(self, sae, normalizer, quantise_fn=None):
        self.sae         = sae
        self.normalizer  = normalizer
        self.quantise_fn = quantise_fn
        self._handle     = None
        self.z_clean: list[torch.Tensor] = []    # full-precision bottleneck
        self.z_quant: list[torch.Tensor] = []    # quantised bottleneck

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            self._fn
        )
        return self

    @torch.no_grad()
    def _fn(self, module, input, output):
        h     = output[0]                        # (batch, seq_len, d_model)
        shape = h.shape
        h_flat = h.reshape(-1, D_MODEL).float()  # (batch*seq_len, d_model)

        # normalize → encode → (optionally quantise) → decode → denormalize
        h_norm = self.normalizer(h_flat)
        z      = self.sae.encode(h_norm)

        z_q = self.quantise_fn(z) if self.quantise_fn is not None else z

        self.z_clean.append(z.cpu().half())
        self.z_quant.append(z_q.cpu().half())

        h_recon = self.normalizer.inverse(self.sae.decode(z_q))
        h_recon = h_recon.reshape(shape).to(h.dtype)

        return (h_recon,) + output[1:]

    def pop_bottlenecks(self):
        z_c = torch.cat(self.z_clean, dim=0).float()
        z_q = torch.cat(self.z_quant, dim=0).float()
        self.z_clean.clear()
        self.z_quant.clear()
        return z_c, z_q

    def remove(self):
        if self._handle:
            self._handle.remove()

@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids


# ── 4. Smoke tests ────────────────────────────────────────────────────────────
def run_smoke_tests(model, tokenizer, sae, normalizer, held_out_ids):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    # ── Test 1: distilgpt2 baseline perplexity ────────────────────────────────
    # Without any patching, perplexity should be reasonable (~20-60 for OWT)
    print("\n[1] Baseline perplexity (no patching)")
    ppl = _perplexity_clean(model, held_out_ids[:200])
    print(f"    perplexity = {ppl:.2f}")
    assert 15 < ppl < 200, f"perplexity {ppl:.2f} is outside expected range"
    print("    ✓")

    # ── Test 2: identity patch (SAE encode→decode, no quantisation) ───────────
    # Perplexity should be slightly higher than baseline (SAE is not perfect),
    # but not dramatically so. A well-trained SAE typically adds 2-10% PPL.
    print("\n[2] Identity patch (SAE reconstruction, no quantisation)")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                  quantise_fn=None)
    overhead = 100 * (ppl_sae / ppl - 1)
    print(f"    perplexity = {ppl_sae:.2f}  ({overhead:+.1f}% vs baseline)")
    assert ppl_sae > ppl, "SAE-patched PPL should be >= clean PPL"
    assert ppl_sae < ppl * 5, "SAE reconstruction is unexpectedly bad"
    print("    ✓")

    # ── Test 3: 8-bit quantisation should be close to identity patch ──────────
    print("\n[3] 8-bit per-tensor quantisation")
    q8 = make_quantise_fn("per_tensor", bits=8)
    ppl_8bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q8)
    print(f"    perplexity = {ppl_8bit:.2f}")
    assert ppl_8bit < ppl * 10, "8-bit PPL is implausibly high — check quantisation"
    print("    ✓")

    # ── Test 4: 2-bit should be worse than 8-bit ─────────────────────────────
    print("\n[4] 2-bit per-tensor quantisation")
    q2 = make_quantise_fn("per_tensor", bits=2)
    ppl_2bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q2)
    print(f"    perplexity = {ppl_2bit:.2f}")
    assert ppl_2bit >= ppl_8bit, "2-bit should be >= 8-bit in perplexity"
    print("    ✓")

    # ── Test 5: SDS sanity check ──────────────────────────────────────────────
    print("\n[5] SDS sanity check")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q8,
                                       held_out_ids[:100])
    U_k = _compute_subspace(Z, k=32)
    sds = _sds(Z, Z_q, U_k)
    print(f"    SDS(k=32, 8-bit) = {sds:.4f}  (expect < 0.5 for 8-bit)")
    assert 0 <= sds <= 1, f"SDS={sds} outside [0,1]"
    print("    ✓")

    # ── Test 6: CKA sanity check ──────────────────────────────────────────────
    print("\n[6] CKA sanity check")
    # CKA of a tensor with itself should be 1.0
    cka_self = _cka(Z[:1000], Z[:1000])
    cka_q    = _cka(Z[:1000], Z_q[:1000])
    print(f"    CKA(Z, Z)   = {cka_self:.4f}  (expect 1.0)")
    print(f"    CKA(Z, Z_q) = {cka_q:.4f}    (expect < 1.0)")
    assert abs(cka_self - 1.0) < 1e-3, f"CKA(self)={cka_self}, expected 1.0"
    assert cka_q <= 1.0
    print("    ✓")

    print("\n✓ all smoke tests passed\n")
    return ppl   # return baseline for reference


# ── 5. Quantisation functions ─────────────────────────────────────────────────
def _uniform_quantise(z: torch.Tensor, delta: torch.Tensor,
                      bits: int) -> torch.Tensor:
    """
    Apply uniform quantisation given a pre-computed step size delta.
    Formula (from assignment key formulas):
        ẑ = clip(round(z / Δ), q_min, q_max)
        z̃ = Δ * ẑ
    """
    q_min = -(2 ** (bits - 1))
    q_max =  (2 ** (bits - 1)) - 1
    z_scaled = z / delta.clamp(min=1e-8)
    z_clipped = torch.clamp(torch.round(z_scaled), q_min, q_max)
    return delta * z_clipped


def _calibrate(z: torch.Tensor, quant_type: str, bits: int) -> torch.Tensor:
    """
    Compute per-tensor or per-feature step size Δ from min/max calibration.
    Per-tensor: one Δ for the whole matrix.
    Per-feature: one Δ per feature dimension (column of z).
    """
    n_levels = 2 ** bits - 1
    if quant_type == "per_tensor":
        delta = (z.max() - z.min()) / n_levels
        return delta.expand(z.shape[-1])   # broadcast to feature dim
    elif quant_type == "per_feature":
        # z: (N, m) — compute min/max along the token dimension
        delta = (z.max(dim=0).values - z.min(dim=0).values) / n_levels
        return delta   # (m,)
    else:
        raise ValueError(f"Unknown quant_type: {quant_type}")


def make_quantise_fn(quant_type: str, bits: int,
                     calibration_z: Optional[torch.Tensor] = None
                     ) -> Callable:
    """
    Returns a quantisation function z → z_q that can be passed to the
    patching hook. Calibration uses the provided tensor if given; otherwise
    calibrates on each batch independently (less accurate but workable for
    smoke tests where we don't have a calibration set yet).
    """
    _delta = None
    if calibration_z is not None:
        _delta = _calibrate(calibration_z, quant_type, bits)

    def quantise_fn(z: torch.Tensor) -> torch.Tensor:
        nonlocal _delta
        delta = _delta if _delta is not None else _calibrate(z, quant_type, bits)
        return _uniform_quantise(z, delta.to(z.device), bits)

    return quantise_fn

@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids


# ── 4. Smoke tests ────────────────────────────────────────────────────────────
def run_smoke_tests(model, tokenizer, sae, normalizer, held_out_ids):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    # ── Test 1: distilgpt2 baseline perplexity ────────────────────────────────
    # Without any patching, perplexity should be reasonable (~20-60 for OWT)
    print("\n[1] Baseline perplexity (no patching)")
    ppl = _perplexity_clean(model, held_out_ids[:200])
    print(f"    perplexity = {ppl:.2f}")
    assert 15 < ppl < 200, f"perplexity {ppl:.2f} is outside expected range"
    print("    ✓")

    # ── Test 2: identity patch (SAE encode→decode, no quantisation) ───────────
    # Perplexity should be slightly higher than baseline (SAE is not perfect),
    # but not dramatically so. A well-trained SAE typically adds 2-10% PPL.
    print("\n[2] Identity patch (SAE reconstruction, no quantisation)")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                  quantise_fn=None)
    overhead = 100 * (ppl_sae / ppl - 1)
    print(f"    perplexity = {ppl_sae:.2f}  ({overhead:+.1f}% vs baseline)")
    assert ppl_sae > ppl, "SAE-patched PPL should be >= clean PPL"
    assert ppl_sae < ppl * 5, "SAE reconstruction is unexpectedly bad"
    print("    ✓")

    # ── Test 3: 8-bit quantisation should be close to identity patch ──────────
    print("\n[3] 8-bit per-tensor quantisation")
    q8 = make_quantise_fn("per_tensor", bits=8)
    ppl_8bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q8)
    print(f"    perplexity = {ppl_8bit:.2f}")
    assert ppl_8bit < ppl * 10, "8-bit PPL is implausibly high — check quantisation"
    print("    ✓")

    # ── Test 4: 2-bit should be worse than 8-bit ─────────────────────────────
    print("\n[4] 2-bit per-tensor quantisation")
    q2 = make_quantise_fn("per_tensor", bits=2)
    ppl_2bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q2)
    print(f"    perplexity = {ppl_2bit:.2f}")
    assert ppl_2bit >= ppl_8bit, "2-bit should be >= 8-bit in perplexity"
    print("    ✓")

    # ── Test 5: SDS sanity check ──────────────────────────────────────────────
    print("\n[5] SDS sanity check")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q8,
                                       held_out_ids[:100])
    U_k = _compute_subspace(Z, k=32)
    sds = _sds(Z, Z_q, U_k)
    print(f"    SDS(k=32, 8-bit) = {sds:.4f}  (expect < 0.5 for 8-bit)")
    assert 0 <= sds <= 1, f"SDS={sds} outside [0,1]"
    print("    ✓")

    # ── Test 6: CKA sanity check ──────────────────────────────────────────────
    print("\n[6] CKA sanity check")
    # CKA of a tensor with itself should be 1.0
    cka_self = _cka(Z[:1000], Z[:1000])
    cka_q    = _cka(Z[:1000], Z_q[:1000])
    print(f"    CKA(Z, Z)   = {cka_self:.4f}  (expect 1.0)")
    print(f"    CKA(Z, Z_q) = {cka_q:.4f}    (expect < 1.0)")
    assert abs(cka_self - 1.0) < 1e-3, f"CKA(self)={cka_self}, expected 1.0"
    assert cka_q <= 1.0
    print("    ✓")

    print("\n✓ all smoke tests passed\n")
    return ppl   # return baseline for reference


# ── 5. Quantisation functions ─────────────────────────────────────────────────
def _uniform_quantise(z: torch.Tensor, delta: torch.Tensor,
                      bits: int) -> torch.Tensor:
    """
    Apply uniform quantisation given a pre-computed step size delta.
    Formula (from assignment key formulas):
        ẑ = clip(round(z / Δ), q_min, q_max)
        z̃ = Δ * ẑ
    """
    q_min = -(2 ** (bits - 1))
    q_max =  (2 ** (bits - 1)) - 1
    z_scaled = z / delta.clamp(min=1e-8)
    z_clipped = torch.clamp(torch.round(z_scaled), q_min, q_max)
    return delta * z_clipped

@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids


# ── 4. Smoke tests ────────────────────────────────────────────────────────────
def run_smoke_tests(model, tokenizer, sae, normalizer, held_out_ids):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    # ── Test 1: distilgpt2 baseline perplexity ────────────────────────────────
    # Without any patching, perplexity should be reasonable (~20-60 for OWT)
    print("\n[1] Baseline perplexity (no patching)")
    ppl = _perplexity_clean(model, held_out_ids[:200])
    print(f"    perplexity = {ppl:.2f}")
    assert 15 < ppl < 200, f"perplexity {ppl:.2f} is outside expected range"
    print("    ✓")

    # ── Test 2: identity patch (SAE encode→decode, no quantisation) ───────────
    # Perplexity should be slightly higher than baseline (SAE is not perfect),
    # but not dramatically so. A well-trained SAE typically adds 2-10% PPL.
    print("\n[2] Identity patch (SAE reconstruction, no quantisation)")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                  quantise_fn=None)
    overhead = 100 * (ppl_sae / ppl - 1)
    print(f"    perplexity = {ppl_sae:.2f}  ({overhead:+.1f}% vs baseline)")
    assert ppl_sae > ppl, "SAE-patched PPL should be >= clean PPL"
    assert ppl_sae < ppl * 5, "SAE reconstruction is unexpectedly bad"
    print("    ✓")

    # ── Test 3: 8-bit quantisation should be close to identity patch ──────────
    print("\n[3] 8-bit per-tensor quantisation")
    q8 = make_quantise_fn("per_tensor", bits=8)
    ppl_8bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q8)
    print(f"    perplexity = {ppl_8bit:.2f}")
    assert ppl_8bit < ppl * 10, "8-bit PPL is implausibly high — check quantisation"
    print("    ✓")

    # ── Test 4: 2-bit should be worse than 8-bit ─────────────────────────────
    print("\n[4] 2-bit per-tensor quantisation")
    q2 = make_quantise_fn("per_tensor", bits=2)
    ppl_2bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q2)
    print(f"    perplexity = {ppl_2bit:.2f}")
    assert ppl_2bit >= ppl_8bit, "2-bit should be >= 8-bit in perplexity"
    print("    ✓")

    # ── Test 5: SDS sanity check ──────────────────────────────────────────────
    print("\n[5] SDS sanity check")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q8,
                                       held_out_ids[:100])
    U_k = _compute_subspace(Z, k=32)
    sds = _sds(Z, Z_q, U_k)
    print(f"    SDS(k=32, 8-bit) = {sds:.4f}  (expect < 0.5 for 8-bit)")
    assert 0 <= sds <= 1, f"SDS={sds} outside [0,1]"
    print("    ✓")

    # ── Test 6: CKA sanity check ──────────────────────────────────────────────
    print("\n[6] CKA sanity check")
    # CKA of a tensor with itself should be 1.0
    cka_self = _cka(Z[:1000], Z[:1000])
    cka_q    = _cka(Z[:1000], Z_q[:1000])
    print(f"    CKA(Z, Z)   = {cka_self:.4f}  (expect 1.0)")
    print(f"    CKA(Z, Z_q) = {cka_q:.4f}    (expect < 1.0)")
    assert abs(cka_self - 1.0) < 1e-3, f"CKA(self)={cka_self}, expected 1.0"
    assert cka_q <= 1.0
    print("    ✓")

    print("\n✓ all smoke tests passed\n")
    return ppl   # return baseline for reference


# ── 5. Quantisation functions ─────────────────────────────────────────────────
def _uniform_quantise(z: torch.Tensor, delta: torch.Tensor,
                      bits: int) -> torch.Tensor:
    """
    Apply uniform quantisation given a pre-computed step size delta.
    Formula (from assignment key formulas):
        ẑ = clip(round(z / Δ), q_min, q_max)
        z̃ = Δ * ẑ
    """
    q_min = -(2 ** (bits - 1))
    q_max =  (2 ** (bits - 1)) - 1
    z_scaled = z / delta.clamp(min=1e-8)
    z_clipped = torch.clamp(torch.round(z_scaled), q_min, q_max)
    return delta * z_clipped


def _calibrate(z: torch.Tensor, quant_type: str, bits: int) -> torch.Tensor:
    """
    Compute per-tensor or per-feature step size Δ from min/max calibration.
    Per-tensor: one Δ for the whole matrix.
    Per-feature: one Δ per feature dimension (column of z).
    """
    n_levels = 2 ** bits - 1
    if quant_type == "per_tensor":
        delta = (z.max() - z.min()) / n_levels
        return delta.expand(z.shape[-1])   # broadcast to feature dim
    elif quant_type == "per_feature":
        # z: (N, m) — compute min/max along the token dimension
        delta = (z.max(dim=0).values - z.min(dim=0).values) / n_levels
        return delta   # (m,)
    else:
        raise ValueError(f"Unknown quant_type: {quant_type}")

@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids


# ── 4. Smoke tests ────────────────────────────────────────────────────────────
def run_smoke_tests(model, tokenizer, sae, normalizer, held_out_ids):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    # ── Test 1: distilgpt2 baseline perplexity ────────────────────────────────
    # Without any patching, perplexity should be reasonable (~20-60 for OWT)
    print("\n[1] Baseline perplexity (no patching)")
    ppl = _perplexity_clean(model, held_out_ids[:200])
    print(f"    perplexity = {ppl:.2f}")
    assert 15 < ppl < 200, f"perplexity {ppl:.2f} is outside expected range"
    print("    ✓")

    # ── Test 2: identity patch (SAE encode→decode, no quantisation) ───────────
    # Perplexity should be slightly higher than baseline (SAE is not perfect),
    # but not dramatically so. A well-trained SAE typically adds 2-10% PPL.
    print("\n[2] Identity patch (SAE reconstruction, no quantisation)")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                  quantise_fn=None)
    overhead = 100 * (ppl_sae / ppl - 1)
    print(f"    perplexity = {ppl_sae:.2f}  ({overhead:+.1f}% vs baseline)")
    assert ppl_sae > ppl, "SAE-patched PPL should be >= clean PPL"
    assert ppl_sae < ppl * 5, "SAE reconstruction is unexpectedly bad"
    print("    ✓")

    # ── Test 3: 8-bit quantisation should be close to identity patch ──────────
    print("\n[3] 8-bit per-tensor quantisation")
    q8 = make_quantise_fn("per_tensor", bits=8)
    ppl_8bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q8)
    print(f"    perplexity = {ppl_8bit:.2f}")
    assert ppl_8bit < ppl * 10, "8-bit PPL is implausibly high — check quantisation"
    print("    ✓")

    # ── Test 4: 2-bit should be worse than 8-bit ─────────────────────────────
    print("\n[4] 2-bit per-tensor quantisation")
    q2 = make_quantise_fn("per_tensor", bits=2)
    ppl_2bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q2)
    print(f"    perplexity = {ppl_2bit:.2f}")
    assert ppl_2bit >= ppl_8bit, "2-bit should be >= 8-bit in perplexity"
    print("    ✓")

    # ── Test 5: SDS sanity check ──────────────────────────────────────────────
    print("\n[5] SDS sanity check")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q8,
                                       held_out_ids[:100])
    U_k = _compute_subspace(Z, k=32)
    sds = _sds(Z, Z_q, U_k)
    print(f"    SDS(k=32, 8-bit) = {sds:.4f}  (expect < 0.5 for 8-bit)")
    assert 0 <= sds <= 1, f"SDS={sds} outside [0,1]"
    print("    ✓")

    # ── Test 6: CKA sanity check ──────────────────────────────────────────────
    print("\n[6] CKA sanity check")
    # CKA of a tensor with itself should be 1.0
    cka_self = _cka(Z[:1000], Z[:1000])
    cka_q    = _cka(Z[:1000], Z_q[:1000])
    print(f"    CKA(Z, Z)   = {cka_self:.4f}  (expect 1.0)")
    print(f"    CKA(Z, Z_q) = {cka_q:.4f}    (expect < 1.0)")
    assert abs(cka_self - 1.0) < 1e-3, f"CKA(self)={cka_self}, expected 1.0"
    assert cka_q <= 1.0
    print("    ✓")

    print("\n✓ all smoke tests passed\n")
    return ppl   # return baseline for reference


# ── 5. Quantisation functions ─────────────────────────────────────────────────
def _uniform_quantise(z: torch.Tensor, delta: torch.Tensor,
                      bits: int) -> torch.Tensor:
    """
    Apply uniform quantisation given a pre-computed step size delta.
    Formula (from assignment key formulas):
        ẑ = clip(round(z / Δ), q_min, q_max)
        z̃ = Δ * ẑ
    """
    q_min = -(2 ** (bits - 1))
    q_max =  (2 ** (bits - 1)) - 1
    z_scaled = z / delta.clamp(min=1e-8)
    z_clipped = torch.clamp(torch.round(z_scaled), q_min, q_max)
    return delta * z_clipped


def _calibrate(z: torch.Tensor, quant_type: str, bits: int) -> torch.Tensor:
    """
    Compute per-tensor or per-feature step size Δ from min/max calibration.
    Per-tensor: one Δ for the whole matrix.
    Per-feature: one Δ per feature dimension (column of z).
    """
    n_levels = 2 ** bits - 1
    if quant_type == "per_tensor":
        delta = (z.max() - z.min()) / n_levels
        return delta.expand(z.shape[-1])   # broadcast to feature dim
    elif quant_type == "per_feature":
        # z: (N, m) — compute min/max along the token dimension
        delta = (z.max(dim=0).values - z.min(dim=0).values) / n_levels
        return delta   # (m,)
    else:
        raise ValueError(f"Unknown quant_type: {quant_type}")


def make_quantise_fn(quant_type: str, bits: int,
                     calibration_z: Optional[torch.Tensor] = None
                     ) -> Callable:
    """
    Returns a quantisation function z → z_q that can be passed to the
    patching hook. Calibration uses the provided tensor if given; otherwise
    calibrates on each batch independently (less accurate but workable for
    smoke tests where we don't have a calibration set yet).
    """
    _delta = None
    if calibration_z is not None:
        _delta = _calibrate(calibration_z, quant_type, bits)

    def quantise_fn(z: torch.Tensor) -> torch.Tensor:
        nonlocal _delta
        delta = _delta if _delta is not None else _calibrate(z, quant_type, bits)
        return _uniform_quantise(z, delta.to(z.device), bits)

    return quantise_fn


# ── 6. Hook infrastructure ────────────────────────────────────────────────────
class CaptureHook:
    """
    Captures layer-3 hidden states without modifying them.
    Used to collect full-precision activations for metric computation.
    """
    def __init__(self):
        self.activations: list[torch.Tensor] = []
        self._handle = None

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            lambda m, inp, out: self.activations.append(
                out[0].detach().cpu().to(torch.float16)
            )
        )
        return self

    def pop(self) -> torch.Tensor:
        out = torch.cat(self.activations, dim=0)
        self.activations.clear()
        return out

    def remove(self):
        if self._handle:
            self._handle.remove()


class PatchingHook:
    """
    Replaces layer-3 hidden states with SAE-reconstructed (optionally
    quantised) activations. The patched activations flow through layers 4-5
    and the LM head, so the final perplexity reflects the information loss
    from quantisation.

    Also stores captured bottleneck activations (before and after quantisation)
    for SDS and CKA computation.
    """
    def __init__(self, sae, normalizer, quantise_fn=None):
        self.sae         = sae
        self.normalizer  = normalizer
        self.quantise_fn = quantise_fn
        self._handle     = None
        self.z_clean: list[torch.Tensor] = []    # full-precision bottleneck
        self.z_quant: list[torch.Tensor] = []    # quantised bottleneck

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            self._fn
        )
        return self

    @torch.no_grad()
    def _fn(self, module, input, output):
        h     = output[0]                        # (batch, seq_len, d_model)
        shape = h.shape
        h_flat = h.reshape(-1, D_MODEL).float()  # (batch*seq_len, d_model)

        # normalize → encode → (optionally quantise) → decode → denormalize
        h_norm = self.normalizer(h_flat)
        z      = self.sae.encode(h_norm)

        z_q = self.quantise_fn(z) if self.quantise_fn is not None else z

        self.z_clean.append(z.cpu().half())
        self.z_quant.append(z_q.cpu().half())

        h_recon = self.normalizer.inverse(self.sae.decode(z_q))
        h_recon = h_recon.reshape(shape).to(h.dtype)

        return (h_recon,) + output[1:]

    def pop_bottlenecks(self):
        z_c = torch.cat(self.z_clean, dim=0).float()
        z_q = torch.cat(self.z_quant, dim=0).float()
        self.z_clean.clear()
        self.z_quant.clear()
        return z_c, z_q

    def remove(self):
        if self._handle:
            self._handle.remove()


# ── 7. Metrics ────────────────────────────────────────────────────────────────
@torch.no_grad()
def _perplexity_clean(model, input_ids: torch.Tensor) -> float:
    """Baseline perplexity with no patching."""
    total_loss, n = 0.0, 0
    for i in range(0, len(input_ids), EVAL_BATCH):
        batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
        loss  = model(batch, labels=batch).loss
        total_loss += loss.item()
        n += 1
    return math.exp(total_loss / n)

@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids


# ── 4. Smoke tests ────────────────────────────────────────────────────────────
def run_smoke_tests(model, tokenizer, sae, normalizer, held_out_ids):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    # ── Test 1: distilgpt2 baseline perplexity ────────────────────────────────
    # Without any patching, perplexity should be reasonable (~20-60 for OWT)
    print("\n[1] Baseline perplexity (no patching)")
    ppl = _perplexity_clean(model, held_out_ids[:200])
    print(f"    perplexity = {ppl:.2f}")
    assert 15 < ppl < 200, f"perplexity {ppl:.2f} is outside expected range"
    print("    ✓")

    # ── Test 2: identity patch (SAE encode→decode, no quantisation) ───────────
    # Perplexity should be slightly higher than baseline (SAE is not perfect),
    # but not dramatically so. A well-trained SAE typically adds 2-10% PPL.
    print("\n[2] Identity patch (SAE reconstruction, no quantisation)")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                  quantise_fn=None)
    overhead = 100 * (ppl_sae / ppl - 1)
    print(f"    perplexity = {ppl_sae:.2f}  ({overhead:+.1f}% vs baseline)")
    assert ppl_sae > ppl, "SAE-patched PPL should be >= clean PPL"
    assert ppl_sae < ppl * 5, "SAE reconstruction is unexpectedly bad"
    print("    ✓")

    # ── Test 3: 8-bit quantisation should be close to identity patch ──────────
    print("\n[3] 8-bit per-tensor quantisation")
    q8 = make_quantise_fn("per_tensor", bits=8)
    ppl_8bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q8)
    print(f"    perplexity = {ppl_8bit:.2f}")
    assert ppl_8bit < ppl * 10, "8-bit PPL is implausibly high — check quantisation"
    print("    ✓")

    # ── Test 4: 2-bit should be worse than 8-bit ─────────────────────────────
    print("\n[4] 2-bit per-tensor quantisation")
    q2 = make_quantise_fn("per_tensor", bits=2)
    ppl_2bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q2)
    print(f"    perplexity = {ppl_2bit:.2f}")
    assert ppl_2bit >= ppl_8bit, "2-bit should be >= 8-bit in perplexity"
    print("    ✓")

    # ── Test 5: SDS sanity check ──────────────────────────────────────────────
    print("\n[5] SDS sanity check")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q8,
                                       held_out_ids[:100])
    U_k = _compute_subspace(Z, k=32)
    sds = _sds(Z, Z_q, U_k)
    print(f"    SDS(k=32, 8-bit) = {sds:.4f}  (expect < 0.5 for 8-bit)")
    assert 0 <= sds <= 1, f"SDS={sds} outside [0,1]"
    print("    ✓")

    # ── Test 6: CKA sanity check ──────────────────────────────────────────────
    print("\n[6] CKA sanity check")
    # CKA of a tensor with itself should be 1.0
    cka_self = _cka(Z[:1000], Z[:1000])
    cka_q    = _cka(Z[:1000], Z_q[:1000])
    print(f"    CKA(Z, Z)   = {cka_self:.4f}  (expect 1.0)")
    print(f"    CKA(Z, Z_q) = {cka_q:.4f}    (expect < 1.0)")
    assert abs(cka_self - 1.0) < 1e-3, f"CKA(self)={cka_self}, expected 1.0"
    assert cka_q <= 1.0
    print("    ✓")

    print("\n✓ all smoke tests passed\n")
    return ppl   # return baseline for reference


# ── 5. Quantisation functions ─────────────────────────────────────────────────
def _uniform_quantise(z: torch.Tensor, delta: torch.Tensor,
                      bits: int) -> torch.Tensor:
    """
    Apply uniform quantisation given a pre-computed step size delta.
    Formula (from assignment key formulas):
        ẑ = clip(round(z / Δ), q_min, q_max)
        z̃ = Δ * ẑ
    """
    q_min = -(2 ** (bits - 1))
    q_max =  (2 ** (bits - 1)) - 1
    z_scaled = z / delta.clamp(min=1e-8)
    z_clipped = torch.clamp(torch.round(z_scaled), q_min, q_max)
    return delta * z_clipped


def _calibrate(z: torch.Tensor, quant_type: str, bits: int) -> torch.Tensor:
    """
    Compute per-tensor or per-feature step size Δ from min/max calibration.
    Per-tensor: one Δ for the whole matrix.
    Per-feature: one Δ per feature dimension (column of z).
    """
    n_levels = 2 ** bits - 1
    if quant_type == "per_tensor":
        delta = (z.max() - z.min()) / n_levels
        return delta.expand(z.shape[-1])   # broadcast to feature dim
    elif quant_type == "per_feature":
        # z: (N, m) — compute min/max along the token dimension
        delta = (z.max(dim=0).values - z.min(dim=0).values) / n_levels
        return delta   # (m,)
    else:
        raise ValueError(f"Unknown quant_type: {quant_type}")


def make_quantise_fn(quant_type: str, bits: int,
                     calibration_z: Optional[torch.Tensor] = None
                     ) -> Callable:
    """
    Returns a quantisation function z → z_q that can be passed to the
    patching hook. Calibration uses the provided tensor if given; otherwise
    calibrates on each batch independently (less accurate but workable for
    smoke tests where we don't have a calibration set yet).
    """
    _delta = None
    if calibration_z is not None:
        _delta = _calibrate(calibration_z, quant_type, bits)

    def quantise_fn(z: torch.Tensor) -> torch.Tensor:
        nonlocal _delta
        delta = _delta if _delta is not None else _calibrate(z, quant_type, bits)
        return _uniform_quantise(z, delta.to(z.device), bits)

    return quantise_fn


# ── 6. Hook infrastructure ────────────────────────────────────────────────────
class CaptureHook:
    """
    Captures layer-3 hidden states without modifying them.
    Used to collect full-precision activations for metric computation.
    """
    def __init__(self):
        self.activations: list[torch.Tensor] = []
        self._handle = None

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            lambda m, inp, out: self.activations.append(
                out[0].detach().cpu().to(torch.float16)
            )
        )
        return self

    def pop(self) -> torch.Tensor:
        out = torch.cat(self.activations, dim=0)
        self.activations.clear()
        return out

    def remove(self):
        if self._handle:
            self._handle.remove()


class PatchingHook:
    """
    Replaces layer-3 hidden states with SAE-reconstructed (optionally
    quantised) activations. The patched activations flow through layers 4-5
    and the LM head, so the final perplexity reflects the information loss
    from quantisation.

    Also stores captured bottleneck activations (before and after quantisation)
    for SDS and CKA computation.
    """
    def __init__(self, sae, normalizer, quantise_fn=None):
        self.sae         = sae
        self.normalizer  = normalizer
        self.quantise_fn = quantise_fn
        self._handle     = None
        self.z_clean: list[torch.Tensor] = []    # full-precision bottleneck
        self.z_quant: list[torch.Tensor] = []    # quantised bottleneck

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            self._fn
        )
        return self

    @torch.no_grad()
    def _fn(self, module, input, output):
        h     = output[0]                        # (batch, seq_len, d_model)
        shape = h.shape
        h_flat = h.reshape(-1, D_MODEL).float()  # (batch*seq_len, d_model)

        # normalize → encode → (optionally quantise) → decode → denormalize
        h_norm = self.normalizer(h_flat)
        z      = self.sae.encode(h_norm)

        z_q = self.quantise_fn(z) if self.quantise_fn is not None else z

        self.z_clean.append(z.cpu().half())
        self.z_quant.append(z_q.cpu().half())

        h_recon = self.normalizer.inverse(self.sae.decode(z_q))
        h_recon = h_recon.reshape(shape).to(h.dtype)

        return (h_recon,) + output[1:]

    def pop_bottlenecks(self):
        z_c = torch.cat(self.z_clean, dim=0).float()
        z_q = torch.cat(self.z_quant, dim=0).float()
        self.z_clean.clear()
        self.z_quant.clear()
        return z_c, z_q

    def remove(self):
        if self._handle:
            self._handle.remove()


# ── 7. Metrics ────────────────────────────────────────────────────────────────
@torch.no_grad()
def _perplexity_clean(model, input_ids: torch.Tensor) -> float:
    """Baseline perplexity with no patching."""
    total_loss, n = 0.0, 0
    for i in range(0, len(input_ids), EVAL_BATCH):
        batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
        loss  = model(batch, labels=batch).loss
        total_loss += loss.item()
        n += 1
    return math.exp(total_loss / n)


@torch.no_grad()
def _perplexity_patched(model, sae, normalizer, input_ids: torch.Tensor,
                        quantise_fn=None) -> float:
    """Perplexity with SAE reconstruction (and optional quantisation) patched in."""
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    total_loss, n = 0.0, 0
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            loss  = model(batch, labels=batch).loss
            total_loss += loss.item()
            n += 1
    finally:
        hook.remove()
    return math.exp(total_loss / n)

@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids


# ── 4. Smoke tests ────────────────────────────────────────────────────────────
def run_smoke_tests(model, tokenizer, sae, normalizer, held_out_ids):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    # ── Test 1: distilgpt2 baseline perplexity ────────────────────────────────
    # Without any patching, perplexity should be reasonable (~20-60 for OWT)
    print("\n[1] Baseline perplexity (no patching)")
    ppl = _perplexity_clean(model, held_out_ids[:200])
    print(f"    perplexity = {ppl:.2f}")
    assert 15 < ppl < 200, f"perplexity {ppl:.2f} is outside expected range"
    print("    ✓")

    # ── Test 2: identity patch (SAE encode→decode, no quantisation) ───────────
    # Perplexity should be slightly higher than baseline (SAE is not perfect),
    # but not dramatically so. A well-trained SAE typically adds 2-10% PPL.
    print("\n[2] Identity patch (SAE reconstruction, no quantisation)")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                  quantise_fn=None)
    overhead = 100 * (ppl_sae / ppl - 1)
    print(f"    perplexity = {ppl_sae:.2f}  ({overhead:+.1f}% vs baseline)")
    assert ppl_sae > ppl, "SAE-patched PPL should be >= clean PPL"
    assert ppl_sae < ppl * 5, "SAE reconstruction is unexpectedly bad"
    print("    ✓")

    # ── Test 3: 8-bit quantisation should be close to identity patch ──────────
    print("\n[3] 8-bit per-tensor quantisation")
    q8 = make_quantise_fn("per_tensor", bits=8)
    ppl_8bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q8)
    print(f"    perplexity = {ppl_8bit:.2f}")
    assert ppl_8bit < ppl * 10, "8-bit PPL is implausibly high — check quantisation"
    print("    ✓")

    # ── Test 4: 2-bit should be worse than 8-bit ─────────────────────────────
    print("\n[4] 2-bit per-tensor quantisation")
    q2 = make_quantise_fn("per_tensor", bits=2)
    ppl_2bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q2)
    print(f"    perplexity = {ppl_2bit:.2f}")
    assert ppl_2bit >= ppl_8bit, "2-bit should be >= 8-bit in perplexity"
    print("    ✓")

    # ── Test 5: SDS sanity check ──────────────────────────────────────────────
    print("\n[5] SDS sanity check")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q8,
                                       held_out_ids[:100])
    U_k = _compute_subspace(Z, k=32)
    sds = _sds(Z, Z_q, U_k)
    print(f"    SDS(k=32, 8-bit) = {sds:.4f}  (expect < 0.5 for 8-bit)")
    assert 0 <= sds <= 1, f"SDS={sds} outside [0,1]"
    print("    ✓")

    # ── Test 6: CKA sanity check ──────────────────────────────────────────────
    print("\n[6] CKA sanity check")
    # CKA of a tensor with itself should be 1.0
    cka_self = _cka(Z[:1000], Z[:1000])
    cka_q    = _cka(Z[:1000], Z_q[:1000])
    print(f"    CKA(Z, Z)   = {cka_self:.4f}  (expect 1.0)")
    print(f"    CKA(Z, Z_q) = {cka_q:.4f}    (expect < 1.0)")
    assert abs(cka_self - 1.0) < 1e-3, f"CKA(self)={cka_self}, expected 1.0"
    assert cka_q <= 1.0
    print("    ✓")

    print("\n✓ all smoke tests passed\n")
    return ppl   # return baseline for reference


# ── 5. Quantisation functions ─────────────────────────────────────────────────
def _uniform_quantise(z: torch.Tensor, delta: torch.Tensor,
                      bits: int) -> torch.Tensor:
    """
    Apply uniform quantisation given a pre-computed step size delta.
    Formula (from assignment key formulas):
        ẑ = clip(round(z / Δ), q_min, q_max)
        z̃ = Δ * ẑ
    """
    q_min = -(2 ** (bits - 1))
    q_max =  (2 ** (bits - 1)) - 1
    z_scaled = z / delta.clamp(min=1e-8)
    z_clipped = torch.clamp(torch.round(z_scaled), q_min, q_max)
    return delta * z_clipped


def _calibrate(z: torch.Tensor, quant_type: str, bits: int) -> torch.Tensor:
    """
    Compute per-tensor or per-feature step size Δ from min/max calibration.
    Per-tensor: one Δ for the whole matrix.
    Per-feature: one Δ per feature dimension (column of z).
    """
    n_levels = 2 ** bits - 1
    if quant_type == "per_tensor":
        delta = (z.max() - z.min()) / n_levels
        return delta.expand(z.shape[-1])   # broadcast to feature dim
    elif quant_type == "per_feature":
        # z: (N, m) — compute min/max along the token dimension
        delta = (z.max(dim=0).values - z.min(dim=0).values) / n_levels
        return delta   # (m,)
    else:
        raise ValueError(f"Unknown quant_type: {quant_type}")


def make_quantise_fn(quant_type: str, bits: int,
                     calibration_z: Optional[torch.Tensor] = None
                     ) -> Callable:
    """
    Returns a quantisation function z → z_q that can be passed to the
    patching hook. Calibration uses the provided tensor if given; otherwise
    calibrates on each batch independently (less accurate but workable for
    smoke tests where we don't have a calibration set yet).
    """
    _delta = None
    if calibration_z is not None:
        _delta = _calibrate(calibration_z, quant_type, bits)

    def quantise_fn(z: torch.Tensor) -> torch.Tensor:
        nonlocal _delta
        delta = _delta if _delta is not None else _calibrate(z, quant_type, bits)
        return _uniform_quantise(z, delta.to(z.device), bits)

    return quantise_fn


# ── 6. Hook infrastructure ────────────────────────────────────────────────────
class CaptureHook:
    """
    Captures layer-3 hidden states without modifying them.
    Used to collect full-precision activations for metric computation.
    """
    def __init__(self):
        self.activations: list[torch.Tensor] = []
        self._handle = None

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            lambda m, inp, out: self.activations.append(
                out[0].detach().cpu().to(torch.float16)
            )
        )
        return self

    def pop(self) -> torch.Tensor:
        out = torch.cat(self.activations, dim=0)
        self.activations.clear()
        return out

    def remove(self):
        if self._handle:
            self._handle.remove()


class PatchingHook:
    """
    Replaces layer-3 hidden states with SAE-reconstructed (optionally
    quantised) activations. The patched activations flow through layers 4-5
    and the LM head, so the final perplexity reflects the information loss
    from quantisation.

    Also stores captured bottleneck activations (before and after quantisation)
    for SDS and CKA computation.
    """
    def __init__(self, sae, normalizer, quantise_fn=None):
        self.sae         = sae
        self.normalizer  = normalizer
        self.quantise_fn = quantise_fn
        self._handle     = None
        self.z_clean: list[torch.Tensor] = []    # full-precision bottleneck
        self.z_quant: list[torch.Tensor] = []    # quantised bottleneck

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            self._fn
        )
        return self

    @torch.no_grad()
    def _fn(self, module, input, output):
        h     = output[0]                        # (batch, seq_len, d_model)
        shape = h.shape
        h_flat = h.reshape(-1, D_MODEL).float()  # (batch*seq_len, d_model)

        # normalize → encode → (optionally quantise) → decode → denormalize
        h_norm = self.normalizer(h_flat)
        z      = self.sae.encode(h_norm)

        z_q = self.quantise_fn(z) if self.quantise_fn is not None else z

        self.z_clean.append(z.cpu().half())
        self.z_quant.append(z_q.cpu().half())

        h_recon = self.normalizer.inverse(self.sae.decode(z_q))
        h_recon = h_recon.reshape(shape).to(h.dtype)

        return (h_recon,) + output[1:]

    def pop_bottlenecks(self):
        z_c = torch.cat(self.z_clean, dim=0).float()
        z_q = torch.cat(self.z_quant, dim=0).float()
        self.z_clean.clear()
        self.z_quant.clear()
        return z_c, z_q

    def remove(self):
        if self._handle:
            self._handle.remove()


# ── 7. Metrics ────────────────────────────────────────────────────────────────
@torch.no_grad()
def _perplexity_clean(model, input_ids: torch.Tensor) -> float:
    """Baseline perplexity with no patching."""
    total_loss, n = 0.0, 0
    for i in range(0, len(input_ids), EVAL_BATCH):
        batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
        loss  = model(batch, labels=batch).loss
        total_loss += loss.item()
        n += 1
    return math.exp(total_loss / n)


@torch.no_grad()
def _perplexity_patched(model, sae, normalizer, input_ids: torch.Tensor,
                        quantise_fn=None) -> float:
    """Perplexity with SAE reconstruction (and optional quantisation) patched in."""
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    total_loss, n = 0.0, 0
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            loss  = model(batch, labels=batch).loss
            total_loss += loss.item()
            n += 1
    finally:
        hook.remove()
    return math.exp(total_loss / n)


@torch.no_grad()
def _collect_bottleneck_pairs(model, sae, normalizer, quantise_fn,
                               input_ids: torch.Tensor):
    """
    Collect full-precision and quantised bottleneck activations on a batch.
    Returns Z (N, m) and Z_q (N, m) in float32.
    """
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            model(batch)   # forward pass; hook captures bottlenecks
    finally:
        hook.remove()
    return hook.pop_bottlenecks()

@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids


# ── 4. Smoke tests ────────────────────────────────────────────────────────────
def run_smoke_tests(model, tokenizer, sae, normalizer, held_out_ids):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    # ── Test 1: distilgpt2 baseline perplexity ────────────────────────────────
    # Without any patching, perplexity should be reasonable (~20-60 for OWT)
    print("\n[1] Baseline perplexity (no patching)")
    ppl = _perplexity_clean(model, held_out_ids[:200])
    print(f"    perplexity = {ppl:.2f}")
    assert 15 < ppl < 200, f"perplexity {ppl:.2f} is outside expected range"
    print("    ✓")

    # ── Test 2: identity patch (SAE encode→decode, no quantisation) ───────────
    # Perplexity should be slightly higher than baseline (SAE is not perfect),
    # but not dramatically so. A well-trained SAE typically adds 2-10% PPL.
    print("\n[2] Identity patch (SAE reconstruction, no quantisation)")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                  quantise_fn=None)
    overhead = 100 * (ppl_sae / ppl - 1)
    print(f"    perplexity = {ppl_sae:.2f}  ({overhead:+.1f}% vs baseline)")
    assert ppl_sae > ppl, "SAE-patched PPL should be >= clean PPL"
    assert ppl_sae < ppl * 5, "SAE reconstruction is unexpectedly bad"
    print("    ✓")

    # ── Test 3: 8-bit quantisation should be close to identity patch ──────────
    print("\n[3] 8-bit per-tensor quantisation")
    q8 = make_quantise_fn("per_tensor", bits=8)
    ppl_8bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q8)
    print(f"    perplexity = {ppl_8bit:.2f}")
    assert ppl_8bit < ppl * 10, "8-bit PPL is implausibly high — check quantisation"
    print("    ✓")

    # ── Test 4: 2-bit should be worse than 8-bit ─────────────────────────────
    print("\n[4] 2-bit per-tensor quantisation")
    q2 = make_quantise_fn("per_tensor", bits=2)
    ppl_2bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q2)
    print(f"    perplexity = {ppl_2bit:.2f}")
    assert ppl_2bit >= ppl_8bit, "2-bit should be >= 8-bit in perplexity"
    print("    ✓")

    # ── Test 5: SDS sanity check ──────────────────────────────────────────────
    print("\n[5] SDS sanity check")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q8,
                                       held_out_ids[:100])
    U_k = _compute_subspace(Z, k=32)
    sds = _sds(Z, Z_q, U_k)
    print(f"    SDS(k=32, 8-bit) = {sds:.4f}  (expect < 0.5 for 8-bit)")
    assert 0 <= sds <= 1, f"SDS={sds} outside [0,1]"
    print("    ✓")

    # ── Test 6: CKA sanity check ──────────────────────────────────────────────
    print("\n[6] CKA sanity check")
    # CKA of a tensor with itself should be 1.0
    cka_self = _cka(Z[:1000], Z[:1000])
    cka_q    = _cka(Z[:1000], Z_q[:1000])
    print(f"    CKA(Z, Z)   = {cka_self:.4f}  (expect 1.0)")
    print(f"    CKA(Z, Z_q) = {cka_q:.4f}    (expect < 1.0)")
    assert abs(cka_self - 1.0) < 1e-3, f"CKA(self)={cka_self}, expected 1.0"
    assert cka_q <= 1.0
    print("    ✓")

    print("\n✓ all smoke tests passed\n")
    return ppl   # return baseline for reference


# ── 5. Quantisation functions ─────────────────────────────────────────────────
def _uniform_quantise(z: torch.Tensor, delta: torch.Tensor,
                      bits: int) -> torch.Tensor:
    """
    Apply uniform quantisation given a pre-computed step size delta.
    Formula (from assignment key formulas):
        ẑ = clip(round(z / Δ), q_min, q_max)
        z̃ = Δ * ẑ
    """
    q_min = -(2 ** (bits - 1))
    q_max =  (2 ** (bits - 1)) - 1
    z_scaled = z / delta.clamp(min=1e-8)
    z_clipped = torch.clamp(torch.round(z_scaled), q_min, q_max)
    return delta * z_clipped


def _calibrate(z: torch.Tensor, quant_type: str, bits: int) -> torch.Tensor:
    """
    Compute per-tensor or per-feature step size Δ from min/max calibration.
    Per-tensor: one Δ for the whole matrix.
    Per-feature: one Δ per feature dimension (column of z).
    """
    n_levels = 2 ** bits - 1
    if quant_type == "per_tensor":
        delta = (z.max() - z.min()) / n_levels
        return delta.expand(z.shape[-1])   # broadcast to feature dim
    elif quant_type == "per_feature":
        # z: (N, m) — compute min/max along the token dimension
        delta = (z.max(dim=0).values - z.min(dim=0).values) / n_levels
        return delta   # (m,)
    else:
        raise ValueError(f"Unknown quant_type: {quant_type}")


def make_quantise_fn(quant_type: str, bits: int,
                     calibration_z: Optional[torch.Tensor] = None
                     ) -> Callable:
    """
    Returns a quantisation function z → z_q that can be passed to the
    patching hook. Calibration uses the provided tensor if given; otherwise
    calibrates on each batch independently (less accurate but workable for
    smoke tests where we don't have a calibration set yet).
    """
    _delta = None
    if calibration_z is not None:
        _delta = _calibrate(calibration_z, quant_type, bits)

    def quantise_fn(z: torch.Tensor) -> torch.Tensor:
        nonlocal _delta
        delta = _delta if _delta is not None else _calibrate(z, quant_type, bits)
        return _uniform_quantise(z, delta.to(z.device), bits)

    return quantise_fn


# ── 6. Hook infrastructure ────────────────────────────────────────────────────
class CaptureHook:
    """
    Captures layer-3 hidden states without modifying them.
    Used to collect full-precision activations for metric computation.
    """
    def __init__(self):
        self.activations: list[torch.Tensor] = []
        self._handle = None

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            lambda m, inp, out: self.activations.append(
                out[0].detach().cpu().to(torch.float16)
            )
        )
        return self

    def pop(self) -> torch.Tensor:
        out = torch.cat(self.activations, dim=0)
        self.activations.clear()
        return out

    def remove(self):
        if self._handle:
            self._handle.remove()


class PatchingHook:
    """
    Replaces layer-3 hidden states with SAE-reconstructed (optionally
    quantised) activations. The patched activations flow through layers 4-5
    and the LM head, so the final perplexity reflects the information loss
    from quantisation.

    Also stores captured bottleneck activations (before and after quantisation)
    for SDS and CKA computation.
    """
    def __init__(self, sae, normalizer, quantise_fn=None):
        self.sae         = sae
        self.normalizer  = normalizer
        self.quantise_fn = quantise_fn
        self._handle     = None
        self.z_clean: list[torch.Tensor] = []    # full-precision bottleneck
        self.z_quant: list[torch.Tensor] = []    # quantised bottleneck

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            self._fn
        )
        return self

    @torch.no_grad()
    def _fn(self, module, input, output):
        h     = output[0]                        # (batch, seq_len, d_model)
        shape = h.shape
        h_flat = h.reshape(-1, D_MODEL).float()  # (batch*seq_len, d_model)

        # normalize → encode → (optionally quantise) → decode → denormalize
        h_norm = self.normalizer(h_flat)
        z      = self.sae.encode(h_norm)

        z_q = self.quantise_fn(z) if self.quantise_fn is not None else z

        self.z_clean.append(z.cpu().half())
        self.z_quant.append(z_q.cpu().half())

        h_recon = self.normalizer.inverse(self.sae.decode(z_q))
        h_recon = h_recon.reshape(shape).to(h.dtype)

        return (h_recon,) + output[1:]

    def pop_bottlenecks(self):
        z_c = torch.cat(self.z_clean, dim=0).float()
        z_q = torch.cat(self.z_quant, dim=0).float()
        self.z_clean.clear()
        self.z_quant.clear()
        return z_c, z_q

    def remove(self):
        if self._handle:
            self._handle.remove()


# ── 7. Metrics ────────────────────────────────────────────────────────────────
@torch.no_grad()
def _perplexity_clean(model, input_ids: torch.Tensor) -> float:
    """Baseline perplexity with no patching."""
    total_loss, n = 0.0, 0
    for i in range(0, len(input_ids), EVAL_BATCH):
        batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
        loss  = model(batch, labels=batch).loss
        total_loss += loss.item()
        n += 1
    return math.exp(total_loss / n)


@torch.no_grad()
def _perplexity_patched(model, sae, normalizer, input_ids: torch.Tensor,
                        quantise_fn=None) -> float:
    """Perplexity with SAE reconstruction (and optional quantisation) patched in."""
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    total_loss, n = 0.0, 0
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            loss  = model(batch, labels=batch).loss
            total_loss += loss.item()
            n += 1
    finally:
        hook.remove()
    return math.exp(total_loss / n)


@torch.no_grad()
def _collect_bottleneck_pairs(model, sae, normalizer, quantise_fn,
                               input_ids: torch.Tensor):
    """
    Collect full-precision and quantised bottleneck activations on a batch.
    Returns Z (N, m) and Z_q (N, m) in float32.
    """
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            model(batch)   # forward pass; hook captures bottlenecks
    finally:
        hook.remove()
    return hook.pop_bottlenecks()


def _compute_subspace(Z: torch.Tensor, k: int) -> torch.Tensor:
    """
    Compute top-k right singular vectors of centred full-precision activations.
    Returns U_k: (m, k) — the subspace basis.
    This is fit once on the full-precision activations and reused for all
    quantised variants, so SDS measures distortion relative to the same basis.
    """
    Z_centred = Z - Z.mean(dim=0, keepdim=True)
    # Use float32 for numerical stability in SVD
    _, _, Vt = torch.linalg.svd(Z_centred.float(), full_matrices=False)
    return Vt[:k].T   # (m, k) — right singular vectors as columns

@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids


# ── 4. Smoke tests ────────────────────────────────────────────────────────────
def run_smoke_tests(model, tokenizer, sae, normalizer, held_out_ids):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    # ── Test 1: distilgpt2 baseline perplexity ────────────────────────────────
    # Without any patching, perplexity should be reasonable (~20-60 for OWT)
    print("\n[1] Baseline perplexity (no patching)")
    ppl = _perplexity_clean(model, held_out_ids[:200])
    print(f"    perplexity = {ppl:.2f}")
    assert 15 < ppl < 200, f"perplexity {ppl:.2f} is outside expected range"
    print("    ✓")

    # ── Test 2: identity patch (SAE encode→decode, no quantisation) ───────────
    # Perplexity should be slightly higher than baseline (SAE is not perfect),
    # but not dramatically so. A well-trained SAE typically adds 2-10% PPL.
    print("\n[2] Identity patch (SAE reconstruction, no quantisation)")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                  quantise_fn=None)
    overhead = 100 * (ppl_sae / ppl - 1)
    print(f"    perplexity = {ppl_sae:.2f}  ({overhead:+.1f}% vs baseline)")
    assert ppl_sae > ppl, "SAE-patched PPL should be >= clean PPL"
    assert ppl_sae < ppl * 5, "SAE reconstruction is unexpectedly bad"
    print("    ✓")

    # ── Test 3: 8-bit quantisation should be close to identity patch ──────────
    print("\n[3] 8-bit per-tensor quantisation")
    q8 = make_quantise_fn("per_tensor", bits=8)
    ppl_8bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q8)
    print(f"    perplexity = {ppl_8bit:.2f}")
    assert ppl_8bit < ppl * 10, "8-bit PPL is implausibly high — check quantisation"
    print("    ✓")

    # ── Test 4: 2-bit should be worse than 8-bit ─────────────────────────────
    print("\n[4] 2-bit per-tensor quantisation")
    q2 = make_quantise_fn("per_tensor", bits=2)
    ppl_2bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q2)
    print(f"    perplexity = {ppl_2bit:.2f}")
    assert ppl_2bit >= ppl_8bit, "2-bit should be >= 8-bit in perplexity"
    print("    ✓")

    # ── Test 5: SDS sanity check ──────────────────────────────────────────────
    print("\n[5] SDS sanity check")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q8,
                                       held_out_ids[:100])
    U_k = _compute_subspace(Z, k=32)
    sds = _sds(Z, Z_q, U_k)
    print(f"    SDS(k=32, 8-bit) = {sds:.4f}  (expect < 0.5 for 8-bit)")
    assert 0 <= sds <= 1, f"SDS={sds} outside [0,1]"
    print("    ✓")

    # ── Test 6: CKA sanity check ──────────────────────────────────────────────
    print("\n[6] CKA sanity check")
    # CKA of a tensor with itself should be 1.0
    cka_self = _cka(Z[:1000], Z[:1000])
    cka_q    = _cka(Z[:1000], Z_q[:1000])
    print(f"    CKA(Z, Z)   = {cka_self:.4f}  (expect 1.0)")
    print(f"    CKA(Z, Z_q) = {cka_q:.4f}    (expect < 1.0)")
    assert abs(cka_self - 1.0) < 1e-3, f"CKA(self)={cka_self}, expected 1.0"
    assert cka_q <= 1.0
    print("    ✓")

    print("\n✓ all smoke tests passed\n")
    return ppl   # return baseline for reference


# ── 5. Quantisation functions ─────────────────────────────────────────────────
def _uniform_quantise(z: torch.Tensor, delta: torch.Tensor,
                      bits: int) -> torch.Tensor:
    """
    Apply uniform quantisation given a pre-computed step size delta.
    Formula (from assignment key formulas):
        ẑ = clip(round(z / Δ), q_min, q_max)
        z̃ = Δ * ẑ
    """
    q_min = -(2 ** (bits - 1))
    q_max =  (2 ** (bits - 1)) - 1
    z_scaled = z / delta.clamp(min=1e-8)
    z_clipped = torch.clamp(torch.round(z_scaled), q_min, q_max)
    return delta * z_clipped


def _calibrate(z: torch.Tensor, quant_type: str, bits: int) -> torch.Tensor:
    """
    Compute per-tensor or per-feature step size Δ from min/max calibration.
    Per-tensor: one Δ for the whole matrix.
    Per-feature: one Δ per feature dimension (column of z).
    """
    n_levels = 2 ** bits - 1
    if quant_type == "per_tensor":
        delta = (z.max() - z.min()) / n_levels
        return delta.expand(z.shape[-1])   # broadcast to feature dim
    elif quant_type == "per_feature":
        # z: (N, m) — compute min/max along the token dimension
        delta = (z.max(dim=0).values - z.min(dim=0).values) / n_levels
        return delta   # (m,)
    else:
        raise ValueError(f"Unknown quant_type: {quant_type}")


def make_quantise_fn(quant_type: str, bits: int,
                     calibration_z: Optional[torch.Tensor] = None
                     ) -> Callable:
    """
    Returns a quantisation function z → z_q that can be passed to the
    patching hook. Calibration uses the provided tensor if given; otherwise
    calibrates on each batch independently (less accurate but workable for
    smoke tests where we don't have a calibration set yet).
    """
    _delta = None
    if calibration_z is not None:
        _delta = _calibrate(calibration_z, quant_type, bits)

    def quantise_fn(z: torch.Tensor) -> torch.Tensor:
        nonlocal _delta
        delta = _delta if _delta is not None else _calibrate(z, quant_type, bits)
        return _uniform_quantise(z, delta.to(z.device), bits)

    return quantise_fn


# ── 6. Hook infrastructure ────────────────────────────────────────────────────
class CaptureHook:
    """
    Captures layer-3 hidden states without modifying them.
    Used to collect full-precision activations for metric computation.
    """
    def __init__(self):
        self.activations: list[torch.Tensor] = []
        self._handle = None

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            lambda m, inp, out: self.activations.append(
                out[0].detach().cpu().to(torch.float16)
            )
        )
        return self

    def pop(self) -> torch.Tensor:
        out = torch.cat(self.activations, dim=0)
        self.activations.clear()
        return out

    def remove(self):
        if self._handle:
            self._handle.remove()


class PatchingHook:
    """
    Replaces layer-3 hidden states with SAE-reconstructed (optionally
    quantised) activations. The patched activations flow through layers 4-5
    and the LM head, so the final perplexity reflects the information loss
    from quantisation.

    Also stores captured bottleneck activations (before and after quantisation)
    for SDS and CKA computation.
    """
    def __init__(self, sae, normalizer, quantise_fn=None):
        self.sae         = sae
        self.normalizer  = normalizer
        self.quantise_fn = quantise_fn
        self._handle     = None
        self.z_clean: list[torch.Tensor] = []    # full-precision bottleneck
        self.z_quant: list[torch.Tensor] = []    # quantised bottleneck

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            self._fn
        )
        return self

    @torch.no_grad()
    def _fn(self, module, input, output):
        h     = output[0]                        # (batch, seq_len, d_model)
        shape = h.shape
        h_flat = h.reshape(-1, D_MODEL).float()  # (batch*seq_len, d_model)

        # normalize → encode → (optionally quantise) → decode → denormalize
        h_norm = self.normalizer(h_flat)
        z      = self.sae.encode(h_norm)

        z_q = self.quantise_fn(z) if self.quantise_fn is not None else z

        self.z_clean.append(z.cpu().half())
        self.z_quant.append(z_q.cpu().half())

        h_recon = self.normalizer.inverse(self.sae.decode(z_q))
        h_recon = h_recon.reshape(shape).to(h.dtype)

        return (h_recon,) + output[1:]

    def pop_bottlenecks(self):
        z_c = torch.cat(self.z_clean, dim=0).float()
        z_q = torch.cat(self.z_quant, dim=0).float()
        self.z_clean.clear()
        self.z_quant.clear()
        return z_c, z_q

    def remove(self):
        if self._handle:
            self._handle.remove()


# ── 7. Metrics ────────────────────────────────────────────────────────────────
@torch.no_grad()
def _perplexity_clean(model, input_ids: torch.Tensor) -> float:
    """Baseline perplexity with no patching."""
    total_loss, n = 0.0, 0
    for i in range(0, len(input_ids), EVAL_BATCH):
        batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
        loss  = model(batch, labels=batch).loss
        total_loss += loss.item()
        n += 1
    return math.exp(total_loss / n)


@torch.no_grad()
def _perplexity_patched(model, sae, normalizer, input_ids: torch.Tensor,
                        quantise_fn=None) -> float:
    """Perplexity with SAE reconstruction (and optional quantisation) patched in."""
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    total_loss, n = 0.0, 0
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            loss  = model(batch, labels=batch).loss
            total_loss += loss.item()
            n += 1
    finally:
        hook.remove()
    return math.exp(total_loss / n)


@torch.no_grad()
def _collect_bottleneck_pairs(model, sae, normalizer, quantise_fn,
                               input_ids: torch.Tensor):
    """
    Collect full-precision and quantised bottleneck activations on a batch.
    Returns Z (N, m) and Z_q (N, m) in float32.
    """
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            model(batch)   # forward pass; hook captures bottlenecks
    finally:
        hook.remove()
    return hook.pop_bottlenecks()


def _compute_subspace(Z: torch.Tensor, k: int) -> torch.Tensor:
    """
    Compute top-k right singular vectors of centred full-precision activations.
    Returns U_k: (m, k) — the subspace basis.
    This is fit once on the full-precision activations and reused for all
    quantised variants, so SDS measures distortion relative to the same basis.
    """
    Z_centred = Z - Z.mean(dim=0, keepdim=True)
    # Use float32 for numerical stability in SVD
    _, _, Vt = torch.linalg.svd(Z_centred.float(), full_matrices=False)
    return Vt[:k].T   # (m, k) — right singular vectors as columns


def _sds(Z: torch.Tensor, Z_q: torch.Tensor, U_k: torch.Tensor) -> float:
    """
    Subspace Distortion Score (SDS), formula from assignment key formulas:
        SDS_k = ||(Z - Ẑ) U_k||²_F  /  ||Z U_k||²_F

    Lower is better. Measures what fraction of quantisation error falls inside
    the information-carrying top-k subspace of the full-precision activations.
    """
    diff      = (Z - Z_q).float() @ U_k.float()   # (N, k)
    numerator = diff.norm(p="fro") ** 2

    proj      = Z.float() @ U_k.float()           # (N, k)
    denom     = proj.norm(p="fro") ** 2

    return (numerator / denom.clamp(min=1e-8)).item()

@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids


# ── 4. Smoke tests ────────────────────────────────────────────────────────────
def run_smoke_tests(model, tokenizer, sae, normalizer, held_out_ids):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    # ── Test 1: distilgpt2 baseline perplexity ────────────────────────────────
    # Without any patching, perplexity should be reasonable (~20-60 for OWT)
    print("\n[1] Baseline perplexity (no patching)")
    ppl = _perplexity_clean(model, held_out_ids[:200])
    print(f"    perplexity = {ppl:.2f}")
    assert 15 < ppl < 200, f"perplexity {ppl:.2f} is outside expected range"
    print("    ✓")

    # ── Test 2: identity patch (SAE encode→decode, no quantisation) ───────────
    # Perplexity should be slightly higher than baseline (SAE is not perfect),
    # but not dramatically so. A well-trained SAE typically adds 2-10% PPL.
    print("\n[2] Identity patch (SAE reconstruction, no quantisation)")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                  quantise_fn=None)
    overhead = 100 * (ppl_sae / ppl - 1)
    print(f"    perplexity = {ppl_sae:.2f}  ({overhead:+.1f}% vs baseline)")
    assert ppl_sae > ppl, "SAE-patched PPL should be >= clean PPL"
    assert ppl_sae < ppl * 5, "SAE reconstruction is unexpectedly bad"
    print("    ✓")

    # ── Test 3: 8-bit quantisation should be close to identity patch ──────────
    print("\n[3] 8-bit per-tensor quantisation")
    q8 = make_quantise_fn("per_tensor", bits=8)
    ppl_8bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q8)
    print(f"    perplexity = {ppl_8bit:.2f}")
    assert ppl_8bit < ppl * 10, "8-bit PPL is implausibly high — check quantisation"
    print("    ✓")

    # ── Test 4: 2-bit should be worse than 8-bit ─────────────────────────────
    print("\n[4] 2-bit per-tensor quantisation")
    q2 = make_quantise_fn("per_tensor", bits=2)
    ppl_2bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q2)
    print(f"    perplexity = {ppl_2bit:.2f}")
    assert ppl_2bit >= ppl_8bit, "2-bit should be >= 8-bit in perplexity"
    print("    ✓")

    # ── Test 5: SDS sanity check ──────────────────────────────────────────────
    print("\n[5] SDS sanity check")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q8,
                                       held_out_ids[:100])
    U_k = _compute_subspace(Z, k=32)
    sds = _sds(Z, Z_q, U_k)
    print(f"    SDS(k=32, 8-bit) = {sds:.4f}  (expect < 0.5 for 8-bit)")
    assert 0 <= sds <= 1, f"SDS={sds} outside [0,1]"
    print("    ✓")

    # ── Test 6: CKA sanity check ──────────────────────────────────────────────
    print("\n[6] CKA sanity check")
    # CKA of a tensor with itself should be 1.0
    cka_self = _cka(Z[:1000], Z[:1000])
    cka_q    = _cka(Z[:1000], Z_q[:1000])
    print(f"    CKA(Z, Z)   = {cka_self:.4f}  (expect 1.0)")
    print(f"    CKA(Z, Z_q) = {cka_q:.4f}    (expect < 1.0)")
    assert abs(cka_self - 1.0) < 1e-3, f"CKA(self)={cka_self}, expected 1.0"
    assert cka_q <= 1.0
    print("    ✓")

    print("\n✓ all smoke tests passed\n")
    return ppl   # return baseline for reference


# ── 5. Quantisation functions ─────────────────────────────────────────────────
def _uniform_quantise(z: torch.Tensor, delta: torch.Tensor,
                      bits: int) -> torch.Tensor:
    """
    Apply uniform quantisation given a pre-computed step size delta.
    Formula (from assignment key formulas):
        ẑ = clip(round(z / Δ), q_min, q_max)
        z̃ = Δ * ẑ
    """
    q_min = -(2 ** (bits - 1))
    q_max =  (2 ** (bits - 1)) - 1
    z_scaled = z / delta.clamp(min=1e-8)
    z_clipped = torch.clamp(torch.round(z_scaled), q_min, q_max)
    return delta * z_clipped


def _calibrate(z: torch.Tensor, quant_type: str, bits: int) -> torch.Tensor:
    """
    Compute per-tensor or per-feature step size Δ from min/max calibration.
    Per-tensor: one Δ for the whole matrix.
    Per-feature: one Δ per feature dimension (column of z).
    """
    n_levels = 2 ** bits - 1
    if quant_type == "per_tensor":
        delta = (z.max() - z.min()) / n_levels
        return delta.expand(z.shape[-1])   # broadcast to feature dim
    elif quant_type == "per_feature":
        # z: (N, m) — compute min/max along the token dimension
        delta = (z.max(dim=0).values - z.min(dim=0).values) / n_levels
        return delta   # (m,)
    else:
        raise ValueError(f"Unknown quant_type: {quant_type}")


def make_quantise_fn(quant_type: str, bits: int,
                     calibration_z: Optional[torch.Tensor] = None
                     ) -> Callable:
    """
    Returns a quantisation function z → z_q that can be passed to the
    patching hook. Calibration uses the provided tensor if given; otherwise
    calibrates on each batch independently (less accurate but workable for
    smoke tests where we don't have a calibration set yet).
    """
    _delta = None
    if calibration_z is not None:
        _delta = _calibrate(calibration_z, quant_type, bits)

    def quantise_fn(z: torch.Tensor) -> torch.Tensor:
        nonlocal _delta
        delta = _delta if _delta is not None else _calibrate(z, quant_type, bits)
        return _uniform_quantise(z, delta.to(z.device), bits)

    return quantise_fn


# ── 6. Hook infrastructure ────────────────────────────────────────────────────
class CaptureHook:
    """
    Captures layer-3 hidden states without modifying them.
    Used to collect full-precision activations for metric computation.
    """
    def __init__(self):
        self.activations: list[torch.Tensor] = []
        self._handle = None

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            lambda m, inp, out: self.activations.append(
                out[0].detach().cpu().to(torch.float16)
            )
        )
        return self

    def pop(self) -> torch.Tensor:
        out = torch.cat(self.activations, dim=0)
        self.activations.clear()
        return out

    def remove(self):
        if self._handle:
            self._handle.remove()


class PatchingHook:
    """
    Replaces layer-3 hidden states with SAE-reconstructed (optionally
    quantised) activations. The patched activations flow through layers 4-5
    and the LM head, so the final perplexity reflects the information loss
    from quantisation.

    Also stores captured bottleneck activations (before and after quantisation)
    for SDS and CKA computation.
    """
    def __init__(self, sae, normalizer, quantise_fn=None):
        self.sae         = sae
        self.normalizer  = normalizer
        self.quantise_fn = quantise_fn
        self._handle     = None
        self.z_clean: list[torch.Tensor] = []    # full-precision bottleneck
        self.z_quant: list[torch.Tensor] = []    # quantised bottleneck

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            self._fn
        )
        return self

    @torch.no_grad()
    def _fn(self, module, input, output):
        h     = output[0]                        # (batch, seq_len, d_model)
        shape = h.shape
        h_flat = h.reshape(-1, D_MODEL).float()  # (batch*seq_len, d_model)

        # normalize → encode → (optionally quantise) → decode → denormalize
        h_norm = self.normalizer(h_flat)
        z      = self.sae.encode(h_norm)

        z_q = self.quantise_fn(z) if self.quantise_fn is not None else z

        self.z_clean.append(z.cpu().half())
        self.z_quant.append(z_q.cpu().half())

        h_recon = self.normalizer.inverse(self.sae.decode(z_q))
        h_recon = h_recon.reshape(shape).to(h.dtype)

        return (h_recon,) + output[1:]

    def pop_bottlenecks(self):
        z_c = torch.cat(self.z_clean, dim=0).float()
        z_q = torch.cat(self.z_quant, dim=0).float()
        self.z_clean.clear()
        self.z_quant.clear()
        return z_c, z_q

    def remove(self):
        if self._handle:
            self._handle.remove()


# ── 7. Metrics ────────────────────────────────────────────────────────────────
@torch.no_grad()
def _perplexity_clean(model, input_ids: torch.Tensor) -> float:
    """Baseline perplexity with no patching."""
    total_loss, n = 0.0, 0
    for i in range(0, len(input_ids), EVAL_BATCH):
        batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
        loss  = model(batch, labels=batch).loss
        total_loss += loss.item()
        n += 1
    return math.exp(total_loss / n)


@torch.no_grad()
def _perplexity_patched(model, sae, normalizer, input_ids: torch.Tensor,
                        quantise_fn=None) -> float:
    """Perplexity with SAE reconstruction (and optional quantisation) patched in."""
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    total_loss, n = 0.0, 0
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            loss  = model(batch, labels=batch).loss
            total_loss += loss.item()
            n += 1
    finally:
        hook.remove()
    return math.exp(total_loss / n)


@torch.no_grad()
def _collect_bottleneck_pairs(model, sae, normalizer, quantise_fn,
                               input_ids: torch.Tensor):
    """
    Collect full-precision and quantised bottleneck activations on a batch.
    Returns Z (N, m) and Z_q (N, m) in float32.
    """
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            model(batch)   # forward pass; hook captures bottlenecks
    finally:
        hook.remove()
    return hook.pop_bottlenecks()


def _compute_subspace(Z: torch.Tensor, k: int) -> torch.Tensor:
    """
    Compute top-k right singular vectors of centred full-precision activations.
    Returns U_k: (m, k) — the subspace basis.
    This is fit once on the full-precision activations and reused for all
    quantised variants, so SDS measures distortion relative to the same basis.
    """
    Z_centred = Z - Z.mean(dim=0, keepdim=True)
    # Use float32 for numerical stability in SVD
    _, _, Vt = torch.linalg.svd(Z_centred.float(), full_matrices=False)
    return Vt[:k].T   # (m, k) — right singular vectors as columns


def _sds(Z: torch.Tensor, Z_q: torch.Tensor, U_k: torch.Tensor) -> float:
    """
    Subspace Distortion Score (SDS), formula from assignment key formulas:
        SDS_k = ||(Z - Ẑ) U_k||²_F  /  ||Z U_k||²_F

    Lower is better. Measures what fraction of quantisation error falls inside
    the information-carrying top-k subspace of the full-precision activations.
    """
    diff      = (Z - Z_q).float() @ U_k.float()   # (N, k)
    numerator = diff.norm(p="fro") ** 2

    proj      = Z.float() @ U_k.float()           # (N, k)
    denom     = proj.norm(p="fro") ** 2

    return (numerator / denom.clamp(min=1e-8)).item()


def _cka(X: torch.Tensor, Y: torch.Tensor) -> float:
    """
    Linear Centered Kernel Alignment between X: (N, p) and Y: (N, q).
    Measures representation similarity; 1.0 = identical, 0.0 = orthogonal.

    We use the HSIC estimator:
        CKA = HSIC(K, L) / sqrt(HSIC(K,K) * HSIC(L,L))
        K = X X^T,  L = Y Y^T  (Gram matrices)
    Centering is applied via the H matrix (H = I - 1/n * 11^T).

    We work on a CPU subset to avoid (N×N) VRAM blowup.
    """
    X, Y = X.float().cpu(), Y.float().cpu()
    n = X.shape[0]

    K = X @ X.T   # (n, n)
    L = Y @ Y.T

    # Centre: K_c = H K H,  H = I - 1/n * 11^T
    col_mean_K = K.mean(dim=0, keepdim=True)
    row_mean_K = K.mean(dim=1, keepdim=True)
    grand_K    = K.mean()
    Kc = K - col_mean_K - row_mean_K + grand_K

    col_mean_L = L.mean(dim=0, keepdim=True)
    row_mean_L = L.mean(dim=1, keepdim=True)
    grand_L    = L.mean()
    Lc = L - col_mean_L - row_mean_L + grand_L

    hsic_kl = (Kc * Lc).sum() / ((n - 1) ** 2)
    hsic_kk = (Kc * Kc).sum() / ((n - 1) ** 2)
    hsic_ll = (Lc * Lc).sum() / ((n - 1) ** 2)

    denom = (hsic_kk * hsic_ll).sqrt().clamp(min=1e-8)
    return (hsic_kl / denom).item()

@torch.no_grad()
def collect_held_out(model, tokenizer, n_seqs: int) -> torch.Tensor:
    """
    Returns input_ids of shape (n_seqs, SEQ_LEN) for perplexity evaluation.
    Collected on CPU; moved to GPU in the eval loop.
    """
    windows = list(_token_windows(tokenizer, max_seqs=n_seqs))
    ids = torch.stack(windows)   # (n_seqs, 128)
    print(f"  held-out ids: {ids.shape}")
    return ids


# ── 4. Smoke tests ────────────────────────────────────────────────────────────
def run_smoke_tests(model, tokenizer, sae, normalizer, held_out_ids):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    # ── Test 1: distilgpt2 baseline perplexity ────────────────────────────────
    # Without any patching, perplexity should be reasonable (~20-60 for OWT)
    print("\n[1] Baseline perplexity (no patching)")
    ppl = _perplexity_clean(model, held_out_ids[:200])
    print(f"    perplexity = {ppl:.2f}")
    assert 15 < ppl < 200, f"perplexity {ppl:.2f} is outside expected range"
    print("    ✓")

    # ── Test 2: identity patch (SAE encode→decode, no quantisation) ───────────
    # Perplexity should be slightly higher than baseline (SAE is not perfect),
    # but not dramatically so. A well-trained SAE typically adds 2-10% PPL.
    print("\n[2] Identity patch (SAE reconstruction, no quantisation)")
    ppl_sae = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                  quantise_fn=None)
    overhead = 100 * (ppl_sae / ppl - 1)
    print(f"    perplexity = {ppl_sae:.2f}  ({overhead:+.1f}% vs baseline)")
    assert ppl_sae > ppl, "SAE-patched PPL should be >= clean PPL"
    assert ppl_sae < ppl * 5, "SAE reconstruction is unexpectedly bad"
    print("    ✓")

    # ── Test 3: 8-bit quantisation should be close to identity patch ──────────
    print("\n[3] 8-bit per-tensor quantisation")
    q8 = make_quantise_fn("per_tensor", bits=8)
    ppl_8bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q8)
    print(f"    perplexity = {ppl_8bit:.2f}")
    assert ppl_8bit < ppl * 10, "8-bit PPL is implausibly high — check quantisation"
    print("    ✓")

    # ── Test 4: 2-bit should be worse than 8-bit ─────────────────────────────
    print("\n[4] 2-bit per-tensor quantisation")
    q2 = make_quantise_fn("per_tensor", bits=2)
    ppl_2bit = _perplexity_patched(model, sae, normalizer, held_out_ids[:200],
                                   quantise_fn=q2)
    print(f"    perplexity = {ppl_2bit:.2f}")
    assert ppl_2bit >= ppl_8bit, "2-bit should be >= 8-bit in perplexity"
    print("    ✓")

    # ── Test 5: SDS sanity check ──────────────────────────────────────────────
    print("\n[5] SDS sanity check")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q8,
                                       held_out_ids[:100])
    U_k = _compute_subspace(Z, k=32)
    sds = _sds(Z, Z_q, U_k)
    print(f"    SDS(k=32, 8-bit) = {sds:.4f}  (expect < 0.5 for 8-bit)")
    assert 0 <= sds <= 1, f"SDS={sds} outside [0,1]"
    print("    ✓")

    # ── Test 6: CKA sanity check ──────────────────────────────────────────────
    print("\n[6] CKA sanity check")
    # CKA of a tensor with itself should be 1.0
    cka_self = _cka(Z[:1000], Z[:1000])
    cka_q    = _cka(Z[:1000], Z_q[:1000])
    print(f"    CKA(Z, Z)   = {cka_self:.4f}  (expect 1.0)")
    print(f"    CKA(Z, Z_q) = {cka_q:.4f}    (expect < 1.0)")
    assert abs(cka_self - 1.0) < 1e-3, f"CKA(self)={cka_self}, expected 1.0"
    assert cka_q <= 1.0
    print("    ✓")

    print("\n✓ all smoke tests passed\n")
    return ppl   # return baseline for reference


# ── 5. Quantisation functions ─────────────────────────────────────────────────
def _uniform_quantise(z: torch.Tensor, delta: torch.Tensor,
                      bits: int) -> torch.Tensor:
    """
    Apply uniform quantisation given a pre-computed step size delta.
    Formula (from assignment key formulas):
        ẑ = clip(round(z / Δ), q_min, q_max)
        z̃ = Δ * ẑ
    """
    q_min = -(2 ** (bits - 1))
    q_max =  (2 ** (bits - 1)) - 1
    z_scaled = z / delta.clamp(min=1e-8)
    z_clipped = torch.clamp(torch.round(z_scaled), q_min, q_max)
    return delta * z_clipped


def _calibrate(z: torch.Tensor, quant_type: str, bits: int) -> torch.Tensor:
    """
    Compute per-tensor or per-feature step size Δ from min/max calibration.
    Per-tensor: one Δ for the whole matrix.
    Per-feature: one Δ per feature dimension (column of z).
    """
    n_levels = 2 ** bits - 1
    if quant_type == "per_tensor":
        delta = (z.max() - z.min()) / n_levels
        return delta.expand(z.shape[-1])   # broadcast to feature dim
    elif quant_type == "per_feature":
        # z: (N, m) — compute min/max along the token dimension
        delta = (z.max(dim=0).values - z.min(dim=0).values) / n_levels
        return delta   # (m,)
    else:
        raise ValueError(f"Unknown quant_type: {quant_type}")


def make_quantise_fn(quant_type: str, bits: int,
                     calibration_z: Optional[torch.Tensor] = None
                     ) -> Callable:
    """
    Returns a quantisation function z → z_q that can be passed to the
    patching hook. Calibration uses the provided tensor if given; otherwise
    calibrates on each batch independently (less accurate but workable for
    smoke tests where we don't have a calibration set yet).
    """
    _delta = None
    if calibration_z is not None:
        _delta = _calibrate(calibration_z, quant_type, bits)

    def quantise_fn(z: torch.Tensor) -> torch.Tensor:
        nonlocal _delta
        delta = _delta if _delta is not None else _calibrate(z, quant_type, bits)
        return _uniform_quantise(z, delta.to(z.device), bits)

    return quantise_fn


# ── 6. Hook infrastructure ────────────────────────────────────────────────────
class CaptureHook:
    """
    Captures layer-3 hidden states without modifying them.
    Used to collect full-precision activations for metric computation.
    """
    def __init__(self):
        self.activations: list[torch.Tensor] = []
        self._handle = None

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            lambda m, inp, out: self.activations.append(
                out[0].detach().cpu().to(torch.float16)
            )
        )
        return self

    def pop(self) -> torch.Tensor:
        out = torch.cat(self.activations, dim=0)
        self.activations.clear()
        return out

    def remove(self):
        if self._handle:
            self._handle.remove()


class PatchingHook:
    """
    Replaces layer-3 hidden states with SAE-reconstructed (optionally
    quantised) activations. The patched activations flow through layers 4-5
    and the LM head, so the final perplexity reflects the information loss
    from quantisation.

    Also stores captured bottleneck activations (before and after quantisation)
    for SDS and CKA computation.
    """
    def __init__(self, sae, normalizer, quantise_fn=None):
        self.sae         = sae
        self.normalizer  = normalizer
        self.quantise_fn = quantise_fn
        self._handle     = None
        self.z_clean: list[torch.Tensor] = []    # full-precision bottleneck
        self.z_quant: list[torch.Tensor] = []    # quantised bottleneck

    def register(self, model, layer_idx=LAYER_IDX):
        self._handle = model.transformer.h[layer_idx].register_forward_hook(
            self._fn
        )
        return self

    @torch.no_grad()
    def _fn(self, module, input, output):
        h     = output[0]                        # (batch, seq_len, d_model)
        shape = h.shape
        h_flat = h.reshape(-1, D_MODEL).float()  # (batch*seq_len, d_model)

        # normalize → encode → (optionally quantise) → decode → denormalize
        h_norm = self.normalizer(h_flat)
        z      = self.sae.encode(h_norm)

        z_q = self.quantise_fn(z) if self.quantise_fn is not None else z

        self.z_clean.append(z.cpu().half())
        self.z_quant.append(z_q.cpu().half())

        h_recon = self.normalizer.inverse(self.sae.decode(z_q))
        h_recon = h_recon.reshape(shape).to(h.dtype)

        return (h_recon,) + output[1:]

    def pop_bottlenecks(self):
        z_c = torch.cat(self.z_clean, dim=0).float()
        z_q = torch.cat(self.z_quant, dim=0).float()
        self.z_clean.clear()
        self.z_quant.clear()
        return z_c, z_q

    def remove(self):
        if self._handle:
            self._handle.remove()


# ── 7. Metrics ────────────────────────────────────────────────────────────────
@torch.no_grad()
def _perplexity_clean(model, input_ids: torch.Tensor) -> float:
    """Baseline perplexity with no patching."""
    total_loss, n = 0.0, 0
    for i in range(0, len(input_ids), EVAL_BATCH):
        batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
        loss  = model(batch, labels=batch).loss
        total_loss += loss.item()
        n += 1
    return math.exp(total_loss / n)


@torch.no_grad()
def _perplexity_patched(model, sae, normalizer, input_ids: torch.Tensor,
                        quantise_fn=None) -> float:
    """Perplexity with SAE reconstruction (and optional quantisation) patched in."""
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    total_loss, n = 0.0, 0
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            loss  = model(batch, labels=batch).loss
            total_loss += loss.item()
            n += 1
    finally:
        hook.remove()
    return math.exp(total_loss / n)


@torch.no_grad()
def _collect_bottleneck_pairs(model, sae, normalizer, quantise_fn,
                               input_ids: torch.Tensor):
    """
    Collect full-precision and quantised bottleneck activations on a batch.
    Returns Z (N, m) and Z_q (N, m) in float32.
    """
    hook = PatchingHook(sae, normalizer, quantise_fn)
    hook.register(model)
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            model(batch)   # forward pass; hook captures bottlenecks
    finally:
        hook.remove()
    return hook.pop_bottlenecks()


def _compute_subspace(Z: torch.Tensor, k: int) -> torch.Tensor:
    """
    Compute top-k right singular vectors of centred full-precision activations.
    Returns U_k: (m, k) — the subspace basis.
    This is fit once on the full-precision activations and reused for all
    quantised variants, so SDS measures distortion relative to the same basis.
    """
    Z_centred = Z - Z.mean(dim=0, keepdim=True)
    # Use float32 for numerical stability in SVD
    _, _, Vt = torch.linalg.svd(Z_centred.float(), full_matrices=False)
    return Vt[:k].T   # (m, k) — right singular vectors as columns


def _sds(Z: torch.Tensor, Z_q: torch.Tensor, U_k: torch.Tensor) -> float:
    """
    Subspace Distortion Score (SDS), formula from assignment key formulas:
        SDS_k = ||(Z - Ẑ) U_k||²_F  /  ||Z U_k||²_F

    Lower is better. Measures what fraction of quantisation error falls inside
    the information-carrying top-k subspace of the full-precision activations.
    """
    diff      = (Z - Z_q).float() @ U_k.float()   # (N, k)
    numerator = diff.norm(p="fro") ** 2

    proj      = Z.float() @ U_k.float()           # (N, k)
    denom     = proj.norm(p="fro") ** 2

    return (numerator / denom.clamp(min=1e-8)).item()


def _cka(X: torch.Tensor, Y: torch.Tensor) -> float:
    """
    Linear Centered Kernel Alignment between X: (N, p) and Y: (N, q).
    Measures representation similarity; 1.0 = identical, 0.0 = orthogonal.

    We use the HSIC estimator:
        CKA = HSIC(K, L) / sqrt(HSIC(K,K) * HSIC(L,L))
        K = X X^T,  L = Y Y^T  (Gram matrices)
    Centering is applied via the H matrix (H = I - 1/n * 11^T).

    We work on a CPU subset to avoid (N×N) VRAM blowup.
    """
    X, Y = X.float().cpu(), Y.float().cpu()
    n = X.shape[0]

    K = X @ X.T   # (n, n)
    L = Y @ Y.T

    # Centre: K_c = H K H,  H = I - 1/n * 11^T
    col_mean_K = K.mean(dim=0, keepdim=True)
    row_mean_K = K.mean(dim=1, keepdim=True)
    grand_K    = K.mean()
    Kc = K - col_mean_K - row_mean_K + grand_K

    col_mean_L = L.mean(dim=0, keepdim=True)
    row_mean_L = L.mean(dim=1, keepdim=True)
    grand_L    = L.mean()
    Lc = L - col_mean_L - row_mean_L + grand_L

    hsic_kl = (Kc * Lc).sum() / ((n - 1) ** 2)
    hsic_kk = (Kc * Kc).sum() / ((n - 1) ** 2)
    hsic_ll = (Lc * Lc).sum() / ((n - 1) ** 2)

    denom = (hsic_kk * hsic_ll).sqrt().clamp(min=1e-8)
    return (hsic_kl / denom).item()


def _mse(Z: torch.Tensor, Z_q: torch.Tensor) -> float:
    return F.mse_loss(Z_q.float(), Z.float()).item()

## 2. Neuron Ranking

In [ ]:
# ── 2. Neuron ranking ─────────────────────────────────────────────────────────
def rank_by_l2(Z: torch.Tensor, Z_q: torch.Tensor) -> torch.Tensor:
    """
    Per-feature mean squared difference between full-precision and quantised
    bottleneck activations. Shape: (m,). Higher = more damaged.
    """
    return (Z - Z_q).pow(2).mean(dim=0)   # (m,)


def rank_by_kl(Z: torch.Tensor, Z_q: torch.Tensor,
               n_bins: int = 50, eps: float = 1e-8) -> torch.Tensor:
    """
    KL(P_full || P_quant) per feature, estimated via histogram.
    The distribution is sparse (many zeros), so we use shared bins that
    cover the full range and apply Laplace smoothing.
    """
    m = Z.shape[1]
    kl = torch.zeros(m)
    Z_np, Zq_np = Z.numpy(), Z_q.numpy()

    for i in range(m):
        zi  = Z_np[:, i]
        zqi = Zq_np[:, i]
        lo  = min(zi.min(), zqi.min())
        hi  = max(zi.max(), zqi.max())
        if hi - lo < 1e-8:
            kl[i] = 0.0
            continue
        bins = np.linspace(lo, hi, n_bins + 1)
        p, _ = np.histogram(zi,  bins=bins)
        q, _ = np.histogram(zqi, bins=bins)
        p = (p + eps) / (p + eps).sum()
        q = (q + eps) / (q + eps).sum()
        kl[i] = float(np.sum(p * np.log(p / q)))

    return kl


def plot_neuron_ranking(l2_scores, kl_scores, out_dir):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    top20_l2 = l2_scores.argsort(descending=True)[:20]
    ax1.bar(range(20), l2_scores[top20_l2].numpy())
    ax1.set_xticks(range(20))
    ax1.set_xticklabels(top20_l2.numpy(), rotation=45, fontsize=8)
    ax1.set_title("Top-20 features by ℓ2 damage (4-bit per-tensor)")
    ax1.set_xlabel("Feature index")
    ax1.set_ylabel("Mean squared difference")

    top20_kl = kl_scores.argsort(descending=True)[:20]
    ax2.bar(range(20), kl_scores[top20_kl].numpy(), color="orange")
    ax2.set_xticks(range(20))
    ax2.set_xticklabels(top20_kl.numpy(), rotation=45, fontsize=8)
    ax2.set_title("Top-20 features by KL divergence (4-bit per-tensor)")
    ax2.set_xlabel("Feature index")
    ax2.set_ylabel("KL divergence")

    plt.tight_layout()
    plt.savefig(out_dir / "neuron_ranking.png", dpi=150)
    plt.show()
    print(f"  saved → neuron_ranking.png")




## 3. Spectral Analysis

In [ ]:
# ── 3. Spectral analysis ──────────────────────────────────────────────────────
def spectral_analysis(Z: torch.Tensor, Z_q: torch.Tensor,
                      k: int = SDS_K, out_dir: Path = OUT_DIR):
    """
    Three required outputs:
      (a) Singular value spectra before and after quantisation
      (b) Principal angles between the two top-k subspaces
      (c) SDS (already implemented — called here for reporting)
    """
    Z_c  = (Z  - Z.mean(0)).float()
    Zq_c = (Z_q - Z_q.mean(0)).float()

    # (a) Singular values
    _, S,  Vt  = torch.linalg.svd(Z_c,  full_matrices=False)
    _, Sq, Vtq = torch.linalg.svd(Zq_c, full_matrices=False)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    n_show = min(100, len(S))
    axes[0].plot(S[:n_show].numpy(),  label="Full precision", linewidth=1.5)
    axes[0].plot(Sq[:n_show].numpy(), label="4-bit quant",    linewidth=1.5, linestyle="--")
    axes[0].set_title("Singular value spectrum")
    axes[0].set_xlabel("Index")
    axes[0].set_ylabel("Singular value")
    axes[0].legend()

    # (b) Principal angles: cos(θ_i) = σ_i(U_k^T Û_k)
    # Vt rows are right singular vectors → Vt[:k].T gives basis columns
    Uk  = Vt[:k].T   # (m, k)
    Ukq = Vtq[:k].T  # (m, k)
    cos_angles = torch.linalg.svdvals(Uk.T @ Ukq)   # (k,)

    axes[1].plot(cos_angles.numpy())
    axes[1].axhline(1.0, linestyle="--", color="gray", alpha=0.5)
    axes[1].set_title(f"Principal angles (cos θ, k={k})")
    axes[1].set_xlabel("Index")
    axes[1].set_ylabel("cos θ  (1 = aligned)")
    axes[1].set_ylim(0, 1.05)

    # (c) Variance explained per PC (for low-variance collapse test, Section 4)
    var_orig  = S.pow(2) / S.pow(2).sum()
    var_quant = Sq.pow(2) / Sq.pow(2).sum()
    cum_orig  = var_orig[:n_show].cumsum(0)
    cum_quant = var_quant[:n_show].cumsum(0)
    axes[2].plot(cum_orig.numpy(),  label="Full precision")
    axes[2].plot(cum_quant.numpy(), label="4-bit quant", linestyle="--")
    axes[2].set_title("Cumulative variance explained")
    axes[2].set_xlabel("# components")
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(out_dir / "spectral_analysis.png", dpi=150)
    plt.show()

    min_cos = cos_angles.min().item()
    mean_cos = cos_angles.mean().item()
    print(f"  principal angles: mean cos θ = {mean_cos:.4f}  "
          f"min cos θ = {min_cos:.4f}")
    print(f"  (1.0 = perfectly aligned, 0.0 = orthogonal)")

    return Uk, Ukq, cos_angles




## 4. Top-Activating Tokens

In [ ]:
# ── 4. Top-activating tokens ──────────────────────────────────────────────────
def top_activating_tokens(model, sae, normalizer, input_ids, tokenizer,
                          feature_indices: list[int], top_n: int = 10):
    """
    For each feature in feature_indices, find the top_n tokens (across the
    held-out set) with the highest SAE activation value, and decode them.
    """
    all_z   = []
    all_ids = []

    hook = PatchingHook(sae, normalizer, quantise_fn=None)
    hook.register(model)
    try:
        for i in range(0, len(input_ids), EVAL_BATCH):
            batch = input_ids[i : i + EVAL_BATCH].to(DEVICE)
            model(batch)
            z_c, _ = hook.pop_bottlenecks()         # (batch*seq_len, m)
            all_z.append(z_c.cpu())
            all_ids.append(input_ids[i : i + EVAL_BATCH]
                           .reshape(-1))             # (batch*seq_len,)
    finally:
        hook.remove()

    Z   = torch.cat(all_z,   dim=0)   # (N, m)
    ids = torch.cat(all_ids, dim=0)   # (N,)

    results = {}
    for feat in feature_indices:
        acts      = Z[:, feat]
        top_idx   = acts.argsort(descending=True)[:top_n]
        top_tokens = ids[top_idx].tolist()
        decoded    = [tokenizer.decode([t]) for t in top_tokens]
        results[feat] = list(zip(acts[top_idx].tolist(), decoded))

    return results

## 5. Ablation Study

In [ ]:
# ── 5. Ablation study ─────────────────────────────────────────────────────────
def ablation_perplexity(model, sae, normalizer, input_ids,
                        feature_indices: list[int]) -> float:
    """Perplexity when the specified SAE features are zeroed out."""
    def ablate_fn(z):
        z = z.clone()
        z[:, feature_indices] = 0.0
        return z

    return _perplexity_patched(model, sae, normalizer, input_ids,
                               quantise_fn=ablate_fn)


def run_ablation_study(model, sae, normalizer, input_ids, baseline_ppl,
                       top_features, random_features, out_dir):
    print("Running ablation study...")
    results = {"feature": [], "type": [], "ppl": [], "delta_pct": []}

    for feat in top_features:
        ppl = ablation_perplexity(model, sae, normalizer, input_ids, [feat])
        delta = 100 * (ppl / baseline_ppl - 1)
        results["feature"].append(feat)
        results["type"].append("top-damaged")
        results["ppl"].append(ppl)
        results["delta_pct"].append(delta)
        print(f"  feature {feat:4d} (top-damaged):  PPL={ppl:.2f}  Δ={delta:+.1f}%")

    for feat in random_features:
        ppl = ablation_perplexity(model, sae, normalizer, input_ids, [feat])
        delta = 100 * (ppl / baseline_ppl - 1)
        results["feature"].append(feat)
        results["type"].append("random")
        results["ppl"].append(ppl)
        results["delta_pct"].append(delta)
        print(f"  feature {feat:4d} (random):       PPL={ppl:.2f}  Δ={delta:+.1f}%")

    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ["#D32F2F" if t == "top-damaged" else "#1976D2"
              for t in results["type"]]
    bars = ax.bar(
        [f"{f}\n({t[:3]})" for f, t in zip(results["feature"], results["type"])],
        results["delta_pct"], color=colors
    )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("PPL change (%)")
    ax.set_title("Ablation study: top-damaged vs random features")
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color="#D32F2F", label="Top-damaged"),
                       Patch(color="#1976D2", label="Random")])
    plt.tight_layout()
    plt.savefig(out_dir / "ablation_study.png", dpi=150)
    plt.show()

    return results




## 6. Alignment Test

In [ ]:
# ── 6. Alignment test ─────────────────────────────────────────────────────────
def alignment_test(l2_scores, jacobian_norms):
    """
    Spearman correlation between the quantisation-damage ranking (ℓ2) and
    the perplexity-impact ranking (Jacobian norms).

    We use Jacobian norms as a proxy for perplexity impact because computing
    individual ablation PPL for all 512 features is prohibitively expensive.
    """
    corr, pval = spearmanr(l2_scores.numpy(), jacobian_norms.numpy())
    print(f"\nAlignment test:")
    print(f"  Spearman ρ = {corr:.4f}  (p = {pval:.4f})")
    print(f"  Interpretation: {'strong' if abs(corr) > 0.5 else 'weak'} alignment "
          f"between quantisation damage and perplexity impact")

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(l2_scores.numpy(), jacobian_norms.numpy(), alpha=0.4, s=10)
    ax.set_xlabel("ℓ2 damage score")
    ax.set_ylabel("Jacobian norm (Fisher importance)")
    ax.set_title(f"Damage vs perplexity impact  (ρ={corr:.3f})")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "alignment_test.png", dpi=150)
    plt.show()

    return corr, pval




## 7. Jacobian Norms

In [ ]:
# ── 7. Jacobian norms ─────────────────────────────────────────────────────────
def compute_jacobian_norms(model, sae, normalizer, input_ids,
                           n_batches: int = JACOBIAN_BATCHES) -> torch.Tensor:
    """
    Approximates Fisher-style importance: E[||∂L/∂z_i||²]^0.5 for each unit i.

    Two-pass per batch:
      Pass 1 (no_grad): capture layer-3 hidden states h, encode → z
      Pass 2 (grad):    create z_leaf, decode → inject via hook → loss → backward

    The gradient flows: z_leaf → SAE decode → normalizer.inverse →
      layer 4-5 of distilgpt2 → LM head → cross-entropy loss.
    Frozen model parameters do not require grad; only z_leaf does, so
    backward() only computes ∂L/∂z_leaf — no model parameter gradients.
    """
    importance = torch.zeros(sae.m)

    for b in range(n_batches):
        batch = input_ids[b * EVAL_BATCH : (b + 1) * EVAL_BATCH].to(DEVICE)

        # ── Pass 1: capture hidden states ────────────────────────────────────
        captured = []
        h1 = model.transformer.h[LAYER_IDX].register_forward_hook(
            lambda m, inp, out: captured.append(out[0].detach())
        )
        with torch.no_grad():
            model(batch)
        h1.remove()

        h = captured[0]   # (batch, seq_len, d_model), float32
        h_flat = h.reshape(-1, D_MODEL)

        with torch.no_grad():
            h_norm = normalizer(h_flat.to(DEVICE))
            z = sae.encode(h_norm)

        # ── Pass 2: differentiable path through SAE ───────────────────────────
        z_leaf = z.detach().requires_grad_(True)
        h_recon_norm = sae.decode(z_leaf)
        h_recon = normalizer.inverse(h_recon_norm).reshape(h.shape)

        # Inject via hook; h_recon is connected to z_leaf in the graph
        recon_iter = iter([h_recon])
        h2 = model.transformer.h[LAYER_IDX].register_forward_hook(
            lambda m, inp, out: (next(recon_iter),) + out[1:]
        )

        output = model(batch, labels=batch)
        loss = output.loss
        h2.remove()

        loss.backward()

        with torch.no_grad():
            # Fisher-style: mean squared gradient per feature across tokens
            importance += z_leaf.grad.pow(2).mean(dim=0).cpu()

        z_leaf.grad = None

    return (importance / n_batches).sqrt()   # (m,) RMS Jacobian norms




## 8. Failure Mode Tests

In [ ]:
# ── 8. Failure mode tests ─────────────────────────────────────────────────────
def test_low_variance_collapse(Z, Z_q_2bit, k=32, out_dir=OUT_DIR):
    """
    Check whether 2-bit quantisation collapses the variance of top-k principal
    directions. Collapse ratio = var_quantised / var_full_precision per PC.
    """
    Z_c   = (Z         - Z.mean(0)).float()
    Zq_c  = (Z_q_2bit  - Z_q_2bit.mean(0)).float()

    _, _, Vt = torch.linalg.svd(Z_c, full_matrices=False)
    Vk = Vt[:k].T   # (m, k)

    var_orig  = (Z_c  @ Vk).var(dim=0)
    var_quant = (Zq_c @ Vk).var(dim=0)
    ratio     = var_quant / var_orig.clamp(min=1e-8)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.bar(range(k), ratio.numpy())
    ax.axhline(1.0, linestyle="--", color="red", alpha=0.7, label="no collapse")
    ax.set_title(f"Variance ratio (2-bit / full-precision) across top-{k} PCs")
    ax.set_xlabel("PC index")
    ax.set_ylabel("Variance ratio")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "low_variance_collapse.png", dpi=150)
    plt.show()

    collapsed = (ratio < 0.5).sum().item()
    print(f"  {collapsed}/{k} principal directions have >50% variance collapse at 2-bit")
    return ratio


def test_sparsity_fragility(Z, Z_q, label="4-bit"):
    """
    Compare L0 and active-feature Jaccard similarity before and after quant.
    Sparsity fragility: quantisation erroneously activates/kills features.
    """
    a_orig  = (Z   > 0)
    a_quant = (Z_q > 0)

    l0_orig  = a_orig.float().sum(dim=1).mean().item()
    l0_quant = a_quant.float().sum(dim=1).mean().item()

    inter   = (a_orig & a_quant).float().sum(dim=1)
    union   = (a_orig | a_quant).float().sum(dim=1)
    jaccard = (inter / union.clamp(min=1)).mean().item()

    # Features erroneously activated / killed per token
    false_pos = (~a_orig &  a_quant).float().sum(dim=1).mean().item()
    false_neg = ( a_orig & ~a_quant).float().sum(dim=1).mean().item()

    print(f"  Sparsity fragility ({label}):")
    print(f"    L0 full-precision : {l0_orig:.1f}")
    print(f"    L0 quantised      : {l0_quant:.1f}")
    print(f"    Jaccard similarity: {jaccard:.4f}  (1.0 = same active set)")
    print(f"    False positives   : {false_pos:.2f} features/token (erroneously activated)")
    print(f"    False negatives   : {false_neg:.2f} features/token (erroneously killed)")

    return l0_orig, l0_quant, jaccard


def test_subspace_rotation(cos_angles, label="4-bit"):
    """
    Summarise principal angle results from spectral_analysis() as a
    rotation test. Small angles (cos ≈ 1) = little rotation.
    """
    print(f"  Subspace rotation ({label}):")
    print(f"    Mean cos θ : {cos_angles.mean():.4f}")
    print(f"    Min  cos θ : {cos_angles.min():.4f}")
    print(f"    Fraction of PCs with cos θ < 0.9: "
          f"{(cos_angles < 0.9).float().mean():.2%}")




## 9. Subspace-Preserving Quantisation

In [ ]:
# ── 9. Subspace-preserving quantisation ───────────────────────────────────────
def make_subspace_quant_fn(U_k: torch.Tensor, calibration_z: torch.Tensor,
                           bits_important: int = 8,
                           bits_residual:  int = 2) -> callable:
    """
    Quantisation function that decomposes z into:
      z_imp = z projected onto top-k subspace  (quantised at bits_important)
      z_res = z - z_imp                         (quantised at bits_residual)

    Why this helps: the top-k subspace contains the information-carrying
    directions (as evidenced by the SDS metric). Preserving that subspace
    at higher precision while coarsely quantising the residual concentrates
    bit-budget where it matters most.
    """
    U_k_gpu  = U_k.to(DEVICE)

    # Pre-compute calibration projections once
    calib_gpu  = calibration_z.to(DEVICE)
    z_imp_c    = calib_gpu @ U_k_gpu @ U_k_gpu.T
    z_res_c    = calib_gpu - z_imp_c
    delta_imp  = _calibrate(z_imp_c, "per_feature", bits_important)
    delta_res  = _calibrate(z_res_c, "per_feature", bits_residual)

    def q_fn(z: torch.Tensor) -> torch.Tensor:
        z_proj  = z @ U_k_gpu
        z_imp   = z_proj @ U_k_gpu.T
        z_res   = z - z_imp
        z_imp_q = _uniform_quantise(z_imp, delta_imp, bits_important)
        z_res_q = _uniform_quantise(z_res, delta_res, bits_residual)
        return z_imp_q + z_res_q

    return q_fn


def run_robust_quant_comparison(model, sae, normalizer, held_out_ids,
                                calib_z, subspaces, baseline_ppl, out_dir):
    """
    Compare subspace-preserving quant (8-bit imp + 2-bit res) against
    standard 4-bit uniform on SDS, PPL, and CKA.
    """
    U_k = subspaces[SDS_K].to(DEVICE)

    q_standard  = make_quantise_fn("per_tensor", 4, calibration_z=calib_z.to(DEVICE))
    q_subspace  = make_subspace_quant_fn(U_k, calib_z)

    rows = []
    for label, q_fn in [("Standard 4-bit",           q_standard),
                         ("Subspace-preserving (8+2)", q_subspace)]:
        ppl = _perplexity_patched(model, sae, normalizer, held_out_ids, q_fn)
        Z, Z_q = _collect_bottleneck_pairs(
            model, sae, normalizer, q_fn,
            held_out_ids[:METRIC_SEQS]
        )
        sds = _sds(Z, Z_q, subspaces[SDS_K])
        cka = _cka(Z[:1000], Z_q[:1000])
        mse = _mse(Z, Z_q)
        rows.append((label, ppl, 100*(ppl/baseline_ppl-1), sds, cka, mse))
        print(f"  {label:<30} PPL={ppl:.2f} ({rows[-1][2]:+.1f}%)  "
              f"SDS={sds:.4f}  CKA={cka:.4f}  MSE={mse:.4f}")

    # Bar chart comparison
    labels = [r[0] for r in rows]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    metrics = [
        ("PPL Δ%", [r[2] for r in rows], "salmon"),
        (f"SDS (k={SDS_K})", [r[3] for r in rows], "steelblue"),
        ("CKA",  [r[4] for r in rows], "mediumseagreen"),
    ]
    for ax, (name, vals, color) in zip(axes, metrics):
        ax.bar(labels, vals, color=color)
        ax.set_title(name)
        ax.tick_params(axis="x", labelsize=8)
    plt.suptitle("Standard 4-bit vs subspace-preserving quantisation")
    plt.tight_layout()
    plt.savefig(out_dir / "robust_quant_comparison.png", dpi=150)
    plt.show()

    return rows




## 10. Smoke Tests

In [ ]:
# ── 10. Smoke tests ───────────────────────────────────────────────────────────
def run_smoke_tests(model, sae, normalizer, tokenizer, input_ids,
                    calib_z, subspaces, baseline_ppl):
    print("=" * 60)
    print("SMOKE TESTS")
    print("=" * 60)

    q4 = make_quantise_fn("per_tensor", 4, calibration_z=calib_z.to(DEVICE))

    # ── Test 1: bottleneck collection ─────────────────────────────────────────
    print("\n[1] Bottleneck collection shape")
    Z, Z_q = _collect_bottleneck_pairs(model, sae, normalizer, q4, input_ids[:20])
    assert Z.shape[1]   == M, f"Z.shape = {Z.shape}"
    assert Z_q.shape[1] == M
    assert Z.shape == Z_q.shape
    print(f"    Z shape: {Z.shape}  ✓")

    # ── Test 2: neuron ranking shapes ─────────────────────────────────────────
    print("\n[2] Neuron ranking")
    l2 = rank_by_l2(Z, Z_q)
    kl = rank_by_kl(Z, Z_q)
    assert l2.shape == (M,) and kl.shape == (M,)
    assert (l2 >= 0).all()
    print(f"    ℓ2 scores: min={l2.min():.5f}  max={l2.max():.5f}  ✓")
    print(f"    KL scores: min={kl.min():.5f}  max={kl.max():.5f}  ✓")

    # ── Test 3: singular values ───────────────────────────────────────────────
    print("\n[3] Singular values")
    Z_c = (Z - Z.mean(0)).float()
    _, S, _ = torch.linalg.svd(Z_c, full_matrices=False)
    assert (S >= 0).all(), "singular values must be non-negative"
    assert (S[:-1] >= S[1:]).all(), "singular values must be non-increasing"
    print(f"    top-5 singular values: {S[:5].numpy().round(2)}  ✓")

    # ── Test 4: ablation changes perplexity ───────────────────────────────────
    print("\n[4] Ablation smoke test")
    top_feat = [l2.argmax().item()]
    ppl_abl  = ablation_perplexity(model, sae, normalizer, input_ids[:50], top_feat)
    print(f"    baseline PPL = {baseline_ppl:.2f}")
    print(f"    ablated PPL  = {ppl_abl:.2f}")
    assert ppl_abl >= baseline_ppl, "ablation should not improve PPL"
    print("    ✓")

    # ── Test 5: Jacobian norms ────────────────────────────────────────────────
    print("\n[5] Jacobian norms (2 batches)")
    jac = compute_jacobian_norms(model, sae, normalizer, input_ids, n_batches=2)
    assert jac.shape == (M,)
    assert (jac >= 0).all()
    assert jac.max() > 0, "all Jacobian norms are zero — gradient not flowing"
    print(f"    Jacobian norms: min={jac.min():.5f}  max={jac.max():.5f}  ✓")

    # ── Test 6: subspace-preserving quantisation ──────────────────────────────
    print("\n[6] Subspace-preserving quantisation")
    U_k = subspaces[SDS_K].to(DEVICE)
    q_sub = make_subspace_quant_fn(U_k, calib_z)
    Z_s, Z_sq = _collect_bottleneck_pairs(model, sae, normalizer, q_sub,
                                          input_ids[:20])
    sds_std = _sds(Z, Z_q,  subspaces[SDS_K])
    sds_sub = _sds(Z_s, Z_sq, subspaces[SDS_K])
    print(f"    SDS standard 4-bit:          {sds_std:.4f}")
    print(f"    SDS subspace-preserving 8+2: {sds_sub:.4f}")
    assert Z_sq.shape == Z_s.shape
    print("    ✓")

    print("\n✓ all smoke tests passed\n")




## Orchestration

In [ ]:
# ── Orchestration ─────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # ── Load ─────────────────────────────────────────────────────────────────
    model, tokenizer = load_frozen_model()
    normalizer       = ActivationNormalizer.load(NOTEBOOK01_DIR / "normalizer.pt")
    sae              = load_sae(NOTEBOOK02_DIR / "sae_m512_ckpt_final.pt")

    held_out_ids = collect_held_out(model, tokenizer, n_seqs=HELD_OUT_SEQS)
    metric_ids   = held_out_ids[:METRIC_SEQS]

    # ── Calibration + subspace bases (recompute from this notebook's held-out)
    cap = CaptureHook().register(model)
    with torch.no_grad():
        for i in range(0, 200, EVAL_BATCH):
            model(metric_ids[i:i+EVAL_BATCH].to(DEVICE))
    cap.remove()
    calib_h    = cap.pop().reshape(-1, D_MODEL).float()
    with torch.no_grad():
        calib_z = sae.encode(normalizer(calib_h.to(DEVICE))).cpu()
    subspaces  = {k: _compute_subspace(calib_z, k) for k in [32, 64, 128]}
    baseline_ppl = _perplexity_clean(model, metric_ids)
    print(f"  baseline PPL = {baseline_ppl:.2f}\n")

    # ── Smoke tests (run this cell first; comment out before committing) ──────
    run_smoke_tests(model, sae, normalizer, tokenizer, metric_ids,
                    calib_z, subspaces, baseline_ppl)

    # ── Collect main bottleneck pairs (4-bit per-tensor for damage analysis) ──
    q4 = make_quantise_fn("per_tensor", 4, calibration_z=calib_z.to(DEVICE))
    q2 = make_quantise_fn("per_tensor", 2, calibration_z=calib_z.to(DEVICE))
    Z,    Z_q4 = _collect_bottleneck_pairs(model, sae, normalizer, q4, metric_ids)
    Z_2, Z_q2  = _collect_bottleneck_pairs(model, sae, normalizer, q2, metric_ids)

    # ── Section 3 ─────────────────────────────────────────────────────────────
    print("=" * 60)
    print("SECTION 3: Representation Damage")
    print("=" * 60)

    l2_scores = rank_by_l2(Z, Z_q4)
    kl_scores = rank_by_kl(Z, Z_q4)
    plot_neuron_ranking(l2_scores, kl_scores, OUT_DIR)

    top_damaged = l2_scores.argsort(descending=True)[:TOP_N_FEATURES].tolist()
    rng         = torch.Generator().manual_seed(42)
    random_feats = torch.randperm(M, generator=rng)[:RANDOM_N_FEATURES].tolist()
    print(f"\n  Top-{TOP_N_FEATURES} damaged features (4-bit): {top_damaged}")
    print(f"  Random features:                          {random_feats}")

    Uk, Ukq, cos_angles = spectral_analysis(Z, Z_q4, k=SDS_K, out_dir=OUT_DIR)

    token_report = top_activating_tokens(model, sae, normalizer, metric_ids,
                                         tokenizer, top_damaged)
    print("\nTop-activating tokens for most damaged features:")
    for feat, items in token_report.items():
        top_tokens = [f"'{tok}' ({act:.2f})" for act, tok in items[:5]]
        print(f"  Feature {feat}: {', '.join(top_tokens)}")

    ablation_results = run_ablation_study(
        model, sae, normalizer, metric_ids, baseline_ppl,
        top_damaged, random_feats, OUT_DIR
    )

    print("\nComputing Jacobian norms for alignment test...")
    jac_norms = compute_jacobian_norms(model, sae, normalizer, metric_ids)
    corr, pval = alignment_test(l2_scores, jac_norms)

    # ── Section 4 ─────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("SECTION 4: Mechanistic Explanation")
    print("=" * 60)

    print("\nFailure mode 1: low-variance collapse")
    vc_ratio = test_low_variance_collapse(Z, Z_q2, k=32, out_dir=OUT_DIR)

    print("\nFailure mode 2: sparsity fragility")
    test_sparsity_fragility(Z, Z_q4, label="4-bit")
    test_sparsity_fragility(Z_2, Z_q2, label="2-bit")

    print("\nFailure mode 3: subspace rotation")
    test_subspace_rotation(cos_angles, label="4-bit")

    print("\nSubspace-preserving quantisation comparison:")
    robust_results = run_robust_quant_comparison(
        model, sae, normalizer, metric_ids,
        calib_z, subspaces, baseline_ppl, OUT_DIR
    )

    # ── Summary ───────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"  Baseline PPL                 : {baseline_ppl:.2f}")
    print(f"  Top-damaged features (4-bit) : {top_damaged}")
    print(f"  Damage–impact alignment ρ    : {corr:.4f}  (p={pval:.4f})")
    print(f"  Outputs saved to {OUT_DIR}")